This is the main experiment part. Here we compare the different OOD detectors using RSLR as the search algorith.

In addition it plots the result and some energy landscape for EMNIST. (Need to remove the exit)

Note in our original run the best detector for BigMNIST, BigEMNIST and Tu Berlin are the same on val and test set but for modelnet10 pc-knn mix is slightly better on the validation set while knn mix was better on the test set.

In [ ]:
#TODO for tu lerin download tu belin dataset and put sketches mat in the correct folder(tu_berlin/sketches_matlab).
#TODO also maybe mention truncation limit of 200 somewhere
from hyper_param.ood import base_prepare

base_prepare.OOD_PARAM_SAMPLERS.keys()

In [ ]:
import copy

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "modelnet10"

default_architecutre_mapping = {
    "mnist": "resnet_small",
    "bigger_mnist": "resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist": "bigger_extended_resnet_small",
    "tu_berlin": "bi_lstm",
    "modelnet10": "pointnetplus",
}

architecture = default_architecutre_mapping[dataset]
budget = None

In [ ]:
from data.get_dataset import get_dataset_info, get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info, path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
x = next(iter(test_loader_transformed))[0]

batch_size = next(iter(train_loader))[0].shape[0]

from src.utils.eval.vis import vis_dataset

vis_dataset(train_loader, val_loader, test_loader_transformed)
from model.train import train_and_get_model, train_or_load_energy_model
from model.get_model import get_network
from src.utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                "comparison_unsupervised")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")

In [ ]:
model = get_network(dataset_info, architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train = f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model, model_dir_path, modelname, train_loader, val_loader, trainer_kwargs={
    "accelerator": "auto",
    "max_epochs": dataset_info.epochs,
    "precision": "16-mixed",
}, load_if_exists=True)



In [ ]:
model.eval().to(device)

In [ ]:
#check main model
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]

from src.utils.transforms.apply import grid_resample
from data.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)
from src.utils.replacer import replace_rotation_transforms_2vec

if dataset == "modelnet10":
    transform_seq = replace_rotation_transforms_2vec(transform_seq)

In [ ]:
print(transform_seq.transformations)

In [ ]:
from model.basic_networks import make_deterministic

make_deterministic(model)



In [ ]:
from data.get_dataset import get_layer_embedding_cache_config, create_layer_embedding_cache

cache_config = get_layer_embedding_cache_config(dataset, architecture, transform_name=None, dataset_info=dataset_info)
train_cache = create_layer_embedding_cache(model, train_loader_no_shuffle, cache_config, embedding_cache_path,
                                           device=device)


In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:

from hyper_param.ood import base_prepare

base_prepare.OOD_PARAM_SAMPLERS.keys()

In [ ]:
detectors = ["knn", "per_class_knn", "knn_mixed", "per_class_knn_mixed", "vim", "dice", "ash", "she", "laplace_mi",
             "laplace_weighted", "trust_score", "openmax", "mahalanobis", "rmd", "class_prototype", "react_all",
             "energy", "per_class_prototype", "laplace_entropy", "adjusted_entropy"]

detectors_parameterless = ["entropy", "MSP", "MaxLogit"]
detectors = detectors + detectors_parameterless
from model.basic_networks import FlexibleResNet

if not isinstance(model, FlexibleResNet):
    #remove ash and react_all as they only work with resnets
    detectors.remove("ash")
    detectors.remove("react_all")
    if "react" not in detectors:
        detectors.append("react")
    detectors.append("ash_last")



In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
model

In [ ]:
from typing import Optional, Any
import os
import json
from itertools import chain
import torch
import optuna
from src.utils.eval.ood_performance import evaluate_confidence_module, save_eval_results

def load_or_run_evaluate_ood_metric(
        confidence_module,
        id_loader,
        ood_loader,
        save_path: Optional[str] = None,
        device: torch.device | str = "cuda",
        show_progress: bool = True,
        overwrite: bool = False,
        base_model: Optional[Any] = None,
):
    """
    Evaluate ID + OOD loaders after forcing OOD labels to be negative.
    Removed the previous condition check.
    """

    def relabeled_ood_loader(loader):
        for x, y in loader:
            # force OOD labels to y < 0
            yield x, torch.full_like(y, -1)

    # relabel OOD loader
    ood_loader = relabeled_ood_loader(ood_loader)

    # If no cache path, evaluate directly
    if not save_path:
        combined_loader = chain(id_loader, ood_loader)
        return evaluate_confidence_module(
            confidence_module=confidence_module,
            data_loader=combined_loader,
            device=device,
            show_progress=show_progress,
        )

    # Load cached result if exists and not overwriting
    if not overwrite and os.path.exists(save_path):
        try:
            with open(save_path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass  # corrupted → recompute

    # Evaluate fresh
    combined_loader = chain(id_loader, ood_loader)
    result = evaluate_confidence_module(
        confidence_module=confidence_module,
        data_loader=combined_loader,
        device=device,
        show_progress=show_progress,
    )

    save_eval_results(save_path, result)
    return result


In [ ]:
#main code block runnign teh comaprision using a smal lstage serch followed by a larger one.

In [ ]:
import os
import json
import torch
import gc
import numpy as np
import copy
from torch.utils.data import DataLoader, Subset
import optuna

from hyper_param.ood.base_prepare import (
    run_ood_study,
    get_default_ood_params,
    get_best_ood_params_from_study,
    create_ood_problem,
)
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from search.random_search import RSLR

search_objective = "search"

n_trials_search = 120
n_trials_search_refine = 15
eval_repeats = 5
if dataset in ["tu_berlin", "modelnet10"]:
    eval_repeats = 10
show_progress = True
top_k = 3

#this here forces the main experiment to not load models from smaller loaders
#as this can lead to overfitting to small loaders which is not desired.
#only the loader on stage 2 the full val set should be stored and reused for final eval(so that they are the same as the best selected model)
allow_loading_models_from_smaller_loaders = False

rng = np.random.default_rng(seed=42)

# Determine report_fraction based on dataset
if dataset in ["mnist", "bigger_mnist", "emnist", "bigger_emnist"]:
    report_fraction_stage1 = 0.25
else:
    report_fraction_stage1 = 0.5

transform_seq_arg = transform_seq

optimizer = RSLR(initial_samples=46, local_runs=2, local_max_steps=3, local_opt_kwargs={"lr": 0.1})
if dataset == "tu_berlin":
    optimizer = RSLR(initial_samples=60, local_runs=1, local_max_steps=0, local_opt_kwargs={"lr": 0.1})

optimizer_search_eval = optimizer

base_results_dir = os.path.join(
    current_path,
    "experiment_files",
    "results",
    "comparison_unsupervised",
    str(dataset),
    str(architecture),
    getattr(dataset_info, "transform_seq_name", "default"),
)
os.makedirs(base_results_dir, exist_ok=True)

model.eval().to(device)

#Prepare validation subsets
val_dataset = val_loader_transformed.dataset
n_samples = len(val_dataset)
subset_size = n_samples // 6

all_indices = rng.permutation(n_samples)

subset_indices_1 = all_indices[:subset_size]
subset_indices_2 = all_indices[subset_size: 2 * subset_size]
subset_indices_3 = all_indices[2 * subset_size: 3 * subset_size]

val_subset_1 = Subset(val_dataset, subset_indices_1)
val_subset_2 = Subset(val_dataset, subset_indices_2)
val_subset_3 = Subset(val_dataset, subset_indices_3)

val_loader_small_1 = DataLoader(
    val_subset_1,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)
val_loader_small_2 = DataLoader(
    val_subset_2,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)
val_loader_small_3 = DataLoader(
    val_subset_3,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)

val_loader_transformed_preshuffled = DataLoader(
    Subset(val_dataset, all_indices),
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)

val_dataset_in = val_loader.dataset

val_loader_small_in_dist = DataLoader(
    Subset(val_dataset_in, subset_indices_1),
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)
val_loader_small_in_dist2 = DataLoader(
    Subset(val_dataset_in, subset_indices_2),
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)

val_loader_small_in_dist3 = DataLoader(
    Subset(val_dataset_in, subset_indices_3),
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)

val_loader_preshuffled_in_dist = DataLoader(
    Subset(val_dataset_in, all_indices),
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers
)

for param in model.parameters():
    param.requires_grad = False

print(f"Each subset: {subset_size} samples ({subset_size / n_samples:.1%} of dataset)")
print("Two disjoint subsets created.\n")

# ---------------- Loop over detectors ----------------
for detector in detectors:
    print(f"\n=== Detector: {detector} ===")
    detector_dir = os.path.join(base_results_dir, detector)
    os.makedirs(detector_dir, exist_ok=True)

    params_path = os.path.join(detector_dir, "best_params.json")
    best_model_path_prefix = os.path.join(detector_dir, "best_model")  # Prefix for model files
    eval_path = os.path.join(detector_dir, "eval_results.json")
    val_path = os.path.join(detector_dir, "val_results.json")

    eval_path_default = os.path.join(detector_dir, "eval_results_default.json")
    val_path_default = os.path.join(detector_dir, "val_results_default.json")
    stage1_params_path = os.path.join(detector_dir, "stage1_best_params.json")
    eval_path_ood = os.path.join(detector_dir, "eval_results_ood.json")
    eval_path_ood_best = os.path.join(detector_dir, "eval_results_ood_best.json")

    if os.path.exists(params_path) and os.path.exists(eval_path) and os.path.exists(val_path) and os.path.exists(
            eval_path_default) and os.path.exists(eval_path_ood) and os.path.exists(val_path_default):
        print(f"[{detector}] Found cached results, skipping.")
        continue

    gc.collect();
    torch.cuda.empty_cache()
    default_params = get_default_ood_params(detector)

    if detector not in detectors_parameterless:
        print(f"[{detector}] Starting two-stage hyperparameter optimization...")
        # Load stage1 params if exist
        best_stage1_candidates = []  # Now list of (score, params, model_paths)
        val_loaders_small = [val_loader_small_1, val_loader_small_2, val_loader_small_3]
        trials_per_loader = [n_trials_search // 3, n_trials_search // 3,
                             n_trials_search - n_trials_search // 3 - n_trials_search // 3]
        val_loadders_small_in_dist = [val_loader_small_in_dist, val_loader_small_in_dist2, val_loader_small_in_dist3]

        for i, (small_loader, n_trials_this_run) in enumerate(zip(val_loaders_small, trials_per_loader), start=1):
            part_params_path = os.path.join(detector_dir, f"stage1_part{i}_best_params.json")
            part_scores_path = os.path.join(detector_dir, f"stage1_part{i}_scores.json")
            part_model_paths_path = os.path.join(detector_dir,
                                                 f"stage1_part{i}_model_paths.json")

            if os.path.exists(part_params_path) and os.path.exists(part_scores_path):
                with open(part_params_path, "r") as f_p, open(part_scores_path, "r") as f_s:
                    part_params = json.load(f_p)
                    part_scores = json.load(f_s)
                part_model_paths = []
                if os.path.exists(part_model_paths_path):
                    with open(part_model_paths_path, "r") as f:
                        part_model_paths = json.load(f)
                else:
                    part_model_paths = [None] * len(part_params)
                print(part_params)
                best_stage1_candidates.extend(zip(part_scores, part_params, part_model_paths))
                print(f"[{detector}] Loaded {len(part_params)} stage1 part {i} candidates from {part_params_path}.")
                continue

            print(f"\n[{detector}] Running on coarse loader {i}/3 ({n_trials_this_run} trials)...")

            objective_kwargs_stage1 = {
                "optimizer": optimizer,
                "model": model,
                "train_cache": train_cache,
                "val_loader": small_loader,
                "transform_seq": transform_seq_arg,
                "dataset_info": dataset_info,
                "architecture": architecture,
                "device": str(device),
                "report_fraction": report_fraction_stage1,
                "repeats": 1,
                "val_id_loader": val_loadders_small_in_dist[i - 1],
                "val_ood_loader": small_loader,
            }

            study_stage1 = run_ood_study(
                study_name=f"{detector}_stage1_part{i}",
                storage_path=None,
                detector_name=detector,
                objective_type=search_objective,
                objective_kwargs=objective_kwargs_stage1,
                n_trials=n_trials_this_run,
                enqueue_params=[copy.deepcopy(default_params)],
            )

            part_params = []
            part_scores = []
            part_model_paths = []
            if study_stage1 is not None:
                # --- FILTER ONLY COMPLETE TRIALS ---
                completed_trials = [
                    t for t in study_stage1.trials
                    if t.state == optuna.trial.TrialState.COMPLETE
                ]
                if not completed_trials:
                    print(f"[{detector}] Warning: No completed trials in subset {i}")
                else:
                    topk_trials = sorted(
                        completed_trials,
                        key=lambda t: t.value,  # only final value
                        reverse=True
                    )[:top_k]

                    print(f"[{detector}] Top-{top_k} parameters from subset {i}:")
                    for rank, t in enumerate(topk_trials, start=1):
                        t_copy = copy.deepcopy(t.params)
                        part_params.append(t_copy)
                        part_scores.append(t.value)
                        print(f"  Rank {rank}: value={t.value:.4f}, params={t_copy}")

                        # --- Save model params if they exist ---
                        if "model_params" in t.user_attrs:
                            model_params = t.user_attrs["model_params"]
                            model_paths = []
                            for model_idx, model_state in enumerate(model_params):
                                model_path = os.path.join(detector_dir,
                                                          f"stage1_part{i}_trial_{t.number}_model_{model_idx}.pt")
                                torch.save(model_state, model_path)
                                model_paths.append(model_path)
                            part_model_paths.append(model_paths)
                            print(f"    -> Saved {len(model_paths)} model(s).")
                        else:
                            part_model_paths.append(None)

            # Save part params, scores, and model paths
            with open(part_params_path, "w") as f:
                json.dump(part_params, f, indent=2)
            with open(part_scores_path, "w") as f:
                json.dump(part_scores, f, indent=2)
            with open(part_model_paths_path, "w") as f:
                json.dump(part_model_paths, f, indent=2)
            print(f"[{detector}] Saved {len(part_params)} stage1 part {i} best params and scores.")
            best_stage1_candidates.extend(zip(part_scores, part_params, part_model_paths))

        print(f"[{detector}] Stage1 completed, total candidates: {len(best_stage1_candidates)}.")

        # === Stage 2 ===
        if os.path.exists(params_path):
            with open(params_path, "r") as f:
                best_params = json.load(f)
            print(f"[{detector}] Loaded best params from {params_path}, skipping stage2.")
        else:
            print(f"\n[{detector}] Stage 2: refine search...")
            objective_kwargs_stage2 = {
                "optimizer": optimizer,
                "model": model,
                "train_cache": train_cache,
                "val_loader": val_loader_transformed_preshuffled,
                "transform_seq": transform_seq_arg,
                "dataset_info": dataset_info,
                "architecture": architecture,
                "device": str(device),
                "report_fraction": 0.1,
                "repeats": 1,
                "val_id_loader": val_loader_preshuffled_in_dist,
                "val_ood_loader": val_loader_transformed_preshuffled,
            }

            # --- Sort candidates by score (descending) ---
            best_stage1_candidates.sort(key=lambda x: x[0], reverse=True)
            sorted_best_params = [p for s, p, m in best_stage1_candidates]
            sorted_model_paths = [m for s, p, m in best_stage1_candidates]

            # --- Enqueue best-first, loading models ---
            enqueue_list = []
            # Add default params first
            enqueue_list.append((copy.deepcopy(default_params), {}))

            # Add sorted candidates with their model states
            for params, model_paths in zip(sorted_best_params, sorted_model_paths):
                user_attrs = {}
                if model_paths and allow_loading_models_from_smaller_loaders:
                    loaded_model_params = [torch.load(mp, map_location="cpu") for mp in model_paths if
                                           os.path.exists(mp)]
                    if loaded_model_params:
                        user_attrs["model_params"] = loaded_model_params
                enqueue_list.append((params, user_attrs))

            print(f"[{detector}] Enqueuing {len(enqueue_list)} parameter sets for Stage 2 refinement (best first):")
            for idx, (p, ua) in enumerate(enqueue_list, start=1):
                print(f"  Enqueued {idx}: {p}" + (" (with model)" if ua else ""))

            study_stage2 = run_ood_study(
                study_name=f"{detector}_stage2",
                storage_path=None,
                detector_name=detector,
                objective_type=search_objective,
                objective_kwargs=objective_kwargs_stage2,
                n_trials=n_trials_search_refine,
                enqueue_params=enqueue_list,
            )

            best_params = (
                get_best_ood_params_from_study(study_stage2)
                if study_stage2
                else (sorted_best_params[0] if sorted_best_params else default_params)
            )

            # Save best model states if available
            if study_stage2 and study_stage2.best_trial and "model_params" in study_stage2.best_trial.user_attrs:
                for model_idx, model_state in enumerate(study_stage2.best_trial.user_attrs["model_params"]):
                    torch.save(model_state, f"{best_model_path_prefix}_{model_idx}.pt")
                print(f"[{detector}] Saved best model states.")

            with open(params_path, "w") as f:
                json.dump(best_params, f, indent=2)
            print(f"[{detector}] Saved best parameters to {params_path}")

        # Final evaluations
        if not os.path.exists(eval_path):
            print(f"[{detector}] Evaluating final configuration with optimized params...")

            final_kwargs = {
                "model": model,
                "train_cache": train_cache,
                "transform_seq": transform_seq_arg,
                "dataset_info": dataset_info,
                "architecture": architecture,
                "device": str(device),
                "val_id_loader": val_loader_preshuffled_in_dist,  # Provide loaders for fitting default params
                "val_ood_loader": val_loader_transformed_preshuffled,
            }

            # Load saved model states for optimized params
            loaded_model_params = []
            i = 0
            while os.path.exists(f"{best_model_path_prefix}_{i}.pt"):
                loaded_model_params.append(torch.load(f"{best_model_path_prefix}_{i}.pt"))
                i += 1
            if len(loaded_model_params) > 0:
                final_kwargs["model_params"] = loaded_model_params
                print(f"Loaded {len(loaded_model_params)} final model states for evaluation.")

            problem = create_ood_problem(
                detector_name=detector,
                params=best_params,
                **final_kwargs,
            )

            metrics = load_or_run_evaluate_confidence_and_search(
                model=model,
                optimizer=optimizer_search_eval,
                problem=problem,
                test_loader=test_loader_transformed,
                save_path=eval_path,
                max_batch_override=dataset_info.batch_size_search,
                show_progress=show_progress,
                repeats=eval_repeats,
                return_per_run=True,
                overwrite=False,
                store_val=False,
                store_correct=True,
            )

            print(
                f"[{detector}] Final Search Accuracy (Optimized): "
                f"{metrics['accuracy_mean']:.4f} ± {metrics['accuracy_std']:.4f}"
            )
        # Result on validation set
        if not os.path.exists(val_path):
            print(f"[{detector}] Evaluating final configuration with optimized params...")

            final_kwargs = {
                "model": model,
                "train_cache": train_cache,
                "transform_seq": transform_seq_arg,
                "dataset_info": dataset_info,
                "architecture": architecture,
                "device": str(device),
                "val_id_loader": val_loader_preshuffled_in_dist,  # Provide loaders for fitting default params
                "val_ood_loader": val_loader_transformed_preshuffled,
            }

            # Load saved model states for optimized params
            loaded_model_params = []
            i = 0
            while os.path.exists(f"{best_model_path_prefix}_{i}.pt"):
                loaded_model_params.append(torch.load(f"{best_model_path_prefix}_{i}.pt"))
                i += 1
            if len(loaded_model_params) > 0:
                final_kwargs["model_params"] = loaded_model_params
                print(f"Loaded {len(loaded_model_params)} final model states for evaluation.")

            problem = create_ood_problem(
                detector_name=detector,
                params=best_params,
                **final_kwargs,
            )

            metrics = load_or_run_evaluate_confidence_and_search(
                model=model,
                optimizer=optimizer_search_eval,
                problem=problem,
                test_loader=val_loader_transformed,
                save_path=val_path,
                max_batch_override=dataset_info.batch_size_search,
                show_progress=show_progress,
                repeats=eval_repeats,
                return_per_run=True,
                overwrite=False,
                store_val=False,
                store_correct=True,
            )

            print(
                f"[{detector}] Validation Search Accuracy (Optimized): "
                f"{metrics['accuracy_mean']:.4f} ± {metrics['accuracy_std']:.4f}"
            )
    if not os.path.exists(eval_path_default):
        print(f"[{detector}] Evaluating final configuration with default params...")

        final_kwargs_default = {
            "model": model,
            "train_cache": train_cache,
            "transform_seq": transform_seq_arg,
            "dataset_info": dataset_info,
            "architecture": architecture,
            "device": str(device),
            "val_id_loader": val_loader_preshuffled_in_dist,  # Provide loaders for fitting default params
            "val_ood_loader": val_loader_transformed_preshuffled,
        }

        problem_default = create_ood_problem(
            detector_name=detector,
            params=default_params,
            **final_kwargs_default,
        )

        metrics_default = load_or_run_evaluate_confidence_and_search(
            model=model,
            optimizer=optimizer_search_eval,
            problem=problem_default,
            test_loader=test_loader_transformed,
            save_path=eval_path_default,
            max_batch_override=dataset_info.batch_size_search,
            show_progress=show_progress,
            repeats=eval_repeats,
            return_per_run=True,
            overwrite=False,
            store_val=False,
            store_correct=True,
        )

        print(
            f"[{detector}] Final Search Accuracy (Default): "
            f"{metrics_default['accuracy_mean']:.4f} ± {metrics_default['accuracy_std']:.4f}"
        )
    if not os.path.exists(val_path_default):
        print(f"[{detector}] Evaluating val configuration with default params...")

        final_kwargs_default = {
            "model": model,
            "train_cache": train_cache,
            "transform_seq": transform_seq_arg,
            "dataset_info": dataset_info,
            "architecture": architecture,
            "device": str(device),
            "val_id_loader": val_loader_preshuffled_in_dist,  # Provide loaders for fitting default params
            "val_ood_loader": val_loader_transformed_preshuffled,
        }

        problem_default = create_ood_problem(
            detector_name=detector,
            params=default_params,
            **final_kwargs_default,
        )

        metrics_default = load_or_run_evaluate_confidence_and_search(
            model=model,
            optimizer=optimizer_search_eval,
            problem=problem_default,
            test_loader=val_loader_transformed,
            save_path=val_path_default,
            max_batch_override=dataset_info.batch_size_search,
            show_progress=show_progress,
            repeats=eval_repeats,
            return_per_run=True,
            overwrite=False,
            store_val=False,
            store_correct=True,
        )

        print(
            f"[{detector}] Final Val Search Accuracy (Default): "
            f"{metrics_default['accuracy_mean']:.4f} ± {metrics_default['accuracy_std']:.4f}"
        )
    if not os.path.exists(eval_path_ood):
        print(f"[{detector}] Evaluating final configuration with default params...")

        final_kwargs_default = {
            "model": model,
            "train_cache": train_cache,
            "transform_seq": transform_seq_arg,
            "dataset_info": dataset_info,
            "architecture": architecture,
            "device": str(device),
            "val_id_loader": val_loader_preshuffled_in_dist,  # Provide loaders for fitting default params
            "val_ood_loader": val_loader_transformed_preshuffled,
        }

        problem_default = create_ood_problem(
            detector_name=detector,
            params=default_params,
            **final_kwargs_default,
        )

        metrics_ood = load_or_run_evaluate_ood_metric(
            confidence_module=problem_default.confidence_module,
            id_loader=test_loader,
            ood_loader=test_loader_transformed,
            save_path=eval_path_ood,
            device=device,
            show_progress=True,
            overwrite=False,
        )
    if not os.path.exists(eval_path_ood_best) and not detector in detectors_parameterless:
        print(f"[{detector}] Evaluating final configuration with optimized params...")

        final_kwargs = {
            "model": model,
            "train_cache": train_cache,
            "transform_seq": transform_seq_arg,
            "dataset_info": dataset_info,
            "architecture": architecture,
            "device": str(device),
            "val_id_loader": val_loader_preshuffled_in_dist,  # Provide loaders for fitting default params
            "val_ood_loader": val_loader_transformed_preshuffled,
        }

        # Load saved model states for optimized params
        loaded_model_params = []
        i = 0
        while os.path.exists(f"{best_model_path_prefix}_{i}.pt"):
            loaded_model_params.append(torch.load(f"{best_model_path_prefix}_{i}.pt"))
            i += 1
        if len(loaded_model_params) > 0:
            final_kwargs["model_params"] = loaded_model_params
            print(f"Loaded {len(loaded_model_params)} final model states for evaluation.")

        problem = create_ood_problem(
            detector_name=detector,
            params=best_params,
            **final_kwargs,
        )

        metrics_ood_best = load_or_run_evaluate_ood_metric(
            confidence_module=problem.confidence_module,
            id_loader=test_loader,
            ood_loader=test_loader_transformed,
            save_path=eval_path_ood_best,
            device=device,
            show_progress=True,
            overwrite=False,
        )

print("\nAll detectors processed successfully.")

In [ ]:

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _extract_auroc(res):
    """
    Robust AUROC extractor: always returns (mean, se) where each is a float or np.nan.
    """
    if res is None:
        return (np.nan, np.nan)

    try:
        # common keys
        for k in ("auroc", "auroc_mean", "roc_auc", "auroc_ood", "auroc_score"):
            if k in res:
                mean = res.get(k)
                std = res.get(k + "_std", np.nan)

                # list-like per-run values
                if isinstance(mean, (list, tuple, np.ndarray)):
                    arr = np.array(mean, dtype=float)
                    if arr.size == 0:
                        return (np.nan, np.nan)
                    mean_v = float(np.nanmean(arr))
                    se_v = float(np.nanstd(arr) / np.sqrt(arr.size))
                    return (mean_v, se_v)

                # scalar mean
                try:
                    mean_v = float(mean)
                except Exception:
                    return (np.nan, np.nan)

                # try to parse std / se
                try:
                    se_v = float(std)
                except Exception:
                    se_v = np.nan

                return (mean_v, se_v)

        # per_run structure
        if "per_run" in res and isinstance(res["per_run"], list):
            arr = []
            for r in res["per_run"]:
                for cand in ("auroc", "roc_auc", "auroc_ood"):
                    if cand in r:
                        try:
                            arr.append(float(r[cand]))
                        except Exception:
                            pass
                        break
            if arr:
                arr = np.array(arr, dtype=float)
                return (float(arr.mean()), float(arr.std() / np.sqrt(arr.size)))
            return (np.nan, np.nan)

        # nested metrics dict
        if "metrics" in res and isinstance(res["metrics"], dict):
            return _extract_auroc(res["metrics"])

    except Exception:
        return (np.nan, np.nan)

    return (np.nan, np.nan)


results = []
for det in detectors:
    path_default = os.path.join(base_results_dir, det, "eval_results_ood.json")
    path_best = os.path.join(base_results_dir, det, "eval_results_ood_best.json")

    res_default = None
    res_best = None
    if os.path.exists(path_default):
        try:
            with open(path_default, "r", encoding="utf-8") as f:
                res_default = json.load(f)
        except Exception:
            res_default = None
    if os.path.exists(path_best):
        try:
            with open(path_best, "r", encoding="utf-8") as f:
                res_best = json.load(f)
        except Exception:
            res_best = None

    mean_def, se_def = _extract_auroc(res_default)
    mean_best, se_best = _extract_auroc(res_best)

    results.append({
        "Detector": det,
        "Default_AUROC": mean_def,
        "Default_SE": se_def,
        "Optimized_AUROC": mean_best,
        "Optimized_SE": se_best,
    })

df = pd.DataFrame(results).set_index("Detector")

# ensure numeric columns (replace nested structures with NaN)
for col in ["Default_AUROC", "Optimized_AUROC", "Default_SE", "Optimized_SE"]:
    df[col] = pd.to_numeric(df[col].apply(lambda x: np.nan if isinstance(x, (list, tuple)) else x), errors="coerce")

# sort by best available AUROC
df["Max_AUROC"] = df[["Default_AUROC", "Optimized_AUROC"]].max(axis=1)
df = df.sort_values("Max_AUROC", ascending=True)  # ascending -> best at top for horizontal bar

# plotting
fig, ax = plt.subplots(figsize=(10, max(6, 0.25 * len(df))))
y = np.arange(len(df))
height = 0.35

ax.barh(y - height / 2, df["Default_AUROC"].values, height, xerr=df["Default_SE"].values, label="Default", capsize=3)
ax.barh(y + height / 2, df["Optimized_AUROC"].values, height, xerr=df["Optimized_SE"].values, label="Optimized",
        capsize=3)

ax.set_yticks(y)
ax.set_yticklabels(df.index)
ax.set_xlabel("AUROC")
ax.set_title("OOD Detection AUROC: Default vs Optimized")
ax.legend()
ax.set_xlim(0.0, 1.0)  # AUROC in [0,1]

# save to figures folder if exists (matches earlier notebook layout)
figure_path = os.path.join(current_path, "experiment_files", "export", "fig", "comparision_unsupervised", dataset,
                           getattr(dataset_info, "transform_seq_name", "default"))
os.makedirs(figure_path, exist_ok=True)
plt.savefig(os.path.join(figure_path, "comparision_detectors_auroc.png"), dpi=200)

In [ ]:
import os
import json
import numpy as np
import pandas as pd


# === LOAD RESULTS ===
def load_metrics(result_dir, detectors):
    results = {}
    for det in detectors:
        eval_path = os.path.join(result_dir, det, "eval_results.json")
        eval_path_default = os.path.join(result_dir, det, "eval_results_default.json")
        print(f"Loading {eval_path} and {eval_path_default}...")

        # Initialize metrics with NaN
        metrics_data = {
            "accuracy_mean_optimized": np.nan,
            "accuracy_std_optimized": np.nan,
            "accuracy_mean_default": np.nan,
            "accuracy_std_default": np.nan,
            "accuracy_se_optimized": np.nan,
            "accuracy_se_default": np.nan,
        }

        # Load optimized results if file exists
        if os.path.exists(eval_path):
            with open(eval_path, "r") as f:
                data = json.load(f)
            metrics_data["accuracy_mean_optimized"] = data.get("accuracy_mean", np.nan)
            metrics_data["accuracy_std_optimized"] = data.get("accuracy_std", np.nan)
            metrics_data["accuracy_se_optimized"] = data.get("accuracy_se", np.nan)

        # Load default results if file exists
        if os.path.exists(eval_path_default):
            with open(eval_path_default, "r") as f:
                data_default = json.load(f)
            metrics_data["accuracy_mean_default"] = data_default.get("accuracy_mean", np.nan)
            metrics_data["accuracy_std_default"] = data_default.get("accuracy_std", np.nan)
            metrics_data["accuracy_se_default"] = data_default.get("accuracy_se", np.nan)

        results[det] = metrics_data

    return results



metrics = load_metrics(base_results_dir, detectors)

# === CREATE COMPARISON TABLE ===
comparison_data = []
for det in detectors:
    m = metrics[det]
    comparison_data.append({
        "Detector": det,
        "Optimized_Accuracy": m["accuracy_mean_optimized"],
        "Optimized_Std": m["accuracy_std_optimized"],
        "Default_Accuracy": m["accuracy_mean_default"],
        "Default_Std": m["accuracy_std_default"],
        "Optimized_SE": m["accuracy_se_optimized"],
        "Default_SE": m["accuracy_se_default"],
    })

df_compare = pd.DataFrame(comparison_data)
print("\n=== OOD Detector Comparison ===")
print(df_compare.to_string(index=False))


In [ ]:
import os
import json
import glob

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

results_base_dir = os.path.join(current_path, "experiment_files", "results", "comparison_unsupervised")

target_datasets = {
    "bigger_mnist": "resnet_small",
    "bigger_emnist": "bigger_extended_resnet_small",
    "tu_berlin": "bi_lstm",
    "modelnet10": "pointnetplus",
}

print(f"Scanning results in: {results_base_dir}\n")
print(f"{'Dataset':<15} | {'Best Detector':<15} | {'Val Acc':<10} | {'Test Acc':<10} | {'Is Truly Best Test?'}")
print("-" * 80)

for dataset, architecture in target_datasets.items():
    dataset_dir = os.path.join(results_base_dir, dataset, architecture)

    if not os.path.exists(dataset_dir):
        print(f"{dataset:<15} | No experimental results found at path.")
        continue

    detector_paths = glob.glob(os.path.join(dataset_dir, "*", "*"))
    dataset_compiled_results = []

    for det_path in detector_paths:
        if not os.path.isdir(det_path):
            continue

        detector_name = os.path.basename(det_path)

        val_path = os.path.join(det_path, "val_results.json")
        val_default_path = os.path.join(det_path, "val_results_default.json")
        eval_path = os.path.join(det_path, "eval_results.json")
        eval_default_path = os.path.join(det_path, "eval_results_default.json")

        best_det_val_acc = -1.0
        best_det_eval_acc = -1.0

        if os.path.exists(val_path) and os.path.exists(eval_path):
            try:
                with open(val_path, "r") as f_val, open(eval_path, "r") as f_eval:
                    val_data = json.load(f_val)
                    eval_data = json.load(f_eval)
                v_acc = val_data.get("accuracy_mean")
                e_acc = eval_data.get("accuracy_mean")
                if v_acc is not None and e_acc is not None and v_acc > best_det_val_acc:
                    best_det_val_acc = v_acc
                    best_det_eval_acc = e_acc
            except Exception:
                pass

        if os.path.exists(val_default_path) and os.path.exists(eval_default_path):
            try:
                with open(val_default_path, "r") as f_val_def, open(eval_default_path, "r") as f_eval_def:
                    val_def_data = json.load(f_val_def)
                    eval_def_data = json.load(f_eval_def)
                v_acc_def = val_def_data.get("accuracy_mean")
                e_acc_def = eval_def_data.get("accuracy_mean")
                if v_acc_def is not None and e_acc_def is not None and v_acc_def > best_det_val_acc:
                    best_det_val_acc = v_acc_def
                    best_det_eval_acc = e_acc_def
            except Exception:
                pass

        if best_det_val_acc != -1.0:
            dataset_compiled_results.append({
                "detector": detector_name,
                "val_acc": best_det_val_acc,
                "eval_acc": best_det_eval_acc
            })

    if not dataset_compiled_results:
        print(f"{dataset:<15} | No valid val/eval JSON files found.")
        continue

    best_by_val = max(dataset_compiled_results, key=lambda x: x["val_acc"])
    best_by_eval = max(dataset_compiled_results, key=lambda x: x["eval_acc"])
    is_best_test = best_by_val["detector"] == best_by_eval["detector"]

    if is_best_test:
        status_str = "YES"
    else:
        status_str = f"NO (Best Test was {best_by_eval['detector']} with {best_by_eval['eval_acc']:.4f})"

    print(
        f"{dataset:<15} | "
        f"{best_by_val['detector']:<15} | "
        f"{best_by_val['val_acc']:.4f}     | "
        f"{best_by_val['eval_acc']:.4f}     | "
        f"{status_str}"
    )

print("\nCross-detector verification complete.")

In [ ]:
from src.utils.eval.vis import plt_setup_latex

W = plt_setup_latex()

In [ ]:
name_map = {
    "single_rmd": "1L-RMD",
    "single_mahalanobis": "1L-MD",
    "rmd": "RMD",
    "mahalanobis": "MD",
    "knn_mixed_faiss": "KNN Mix (Faiss)",
    "knn_mixed": "KNN Mix",
    "nn_guided_one": "NN-Guided (1)",

    "knn": "KNN",
    "per_class_knn_mixed": "PC-kNN Mix",
    "per_class_knn": "PC-kNN",
    "knn_itf": "KNN-ITF",

    "trust_score": "Trust Score",
    "vim": "VIM",

    "per_class_prototype": "PC-Proto",
    "prototype_multi": "Prototype Mul.",
    "class_prototype": "Proto",

    "adjusted_entropy": "GEN",
    "laplace_entropy": "Lapl. Entropy",
    "laplace_entropy_gridsearch": "Lapl. Entropy (GS)",
    "react_all": "ReAct",
    "entropy": "Entropy",
    "laplace_weighted": "Lapl. Weighted",

    "react": "ReAct",
    "dice": "DICE",
    "laplace_mi": "Lapl. MI",
    "MaxLogit": "MaxLogit",

    "nn_guided": "NN-Guidance",
    "ash": "ASH",
    "ash_last": "ASH",
    "openmax": "OpenMax",
    "energy_ts": "Energy T. Scale",
    "energy": "Energy",
    "laplace_energy": "Lapl. Energy",
    "mahalanobis_individual": "MD-Diag",
    "single_rmd_individual": "1L-RMD-Diag",
    "single_mahalanobis_individual": "1L-MD-Diag",
    "she": "SHE",
    "rmd_individual": "RMD-Diag",
}




In [ ]:
figure_path = os.path.join(current_path, "experiment_files", "export", "fig", "comparision_unsupervised", dataset,
                           transform_name)
if not os.path.exists(figure_path):
    os.makedirs(figure_path)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Prepare and rename detectors
df_plot = df_compare.assign(
    Max_Accuracy=df_compare[["Default_Accuracy", "Optimized_Accuracy"]].max(axis=1)
).sort_values("Max_Accuracy", ascending=True)

df_plot["Detector"] = df_plot["Detector"].replace(name_map)

y = np.arange(len(df_plot))
height = 0.3

fig, ax = plt.subplots(figsize=(W / 2.0, W * 0.75))

bars1 = ax.barh(
    y - height / 2,
    df_plot["Default_Accuracy"],
    height,
    xerr=df_plot["Default_SE"] * 1.96,
    label="Default",
    capsize=0.8,
    error_kw=dict(lw=0.5, capthick=0.5), zorder=2
)

bars2 = ax.barh(
    y + height / 2,
    df_plot["Optimized_Accuracy"],
    height,
    xerr=df_plot["Optimized_SE"] * 1.96,
    label="Tuned",
    capsize=0.8,
    error_kw=dict(lw=0.5, capthick=0.5), zorder=2
)

ax.set_ylabel("Detector", fontsize=9)
ax.set_xlabel("Accuracy", fontsize=9)

#ax.set_xticklabels(ax.get_xticklabels(), fontsize=8)
ax.tick_params(axis='x', labelsize=8)

ax.margins(y=0.01)

ax.set_yticks(y)
ax.set_yticklabels(df_plot["Detector"], fontsize=7)

if dataset == "modelnet10":
    legend_loc = "lower right"
else:
    legend_loc = "upper left"

ax.legend(fontsize=8, framealpha=0.4, loc=legend_loc)

ax.grid(axis="x", linestyle="--", alpha=0.6, zorder=-1)

#Zoom in X-axis to focus on differences
x_min = min(df_plot["Default_Accuracy"].min(), df_plot["Optimized_Accuracy"].min())
x_max = max(df_plot["Default_Accuracy"].max(), df_plot["Optimized_Accuracy"].max())
margin = (x_max - x_min) * 0.05
ax.set_xlim(x_min - margin, x_max + margin)

figure_path = os.path.join(current_path, "experiment_files", "export", "fig", "comparision_unsupervised", dataset,
                           transform_name)
if not os.path.exists(figure_path):
    os.makedirs(figure_path)
plt.tight_layout()
plt.savefig(os.path.join(figure_path, "comparision_detectors.pdf"), bbox_inches="tight")
plt.savefig(os.path.join(figure_path, "comparision_detectors.pgf"), bbox_inches="tight")

plt.show()



In [ ]:
df_plot

In [ ]:

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# merge AUROC (df) and accuracy table (df_compare)
auroc_df = df.reset_index()[["Detector", "Default_AUROC", "Default_SE", "Optimized_AUROC", "Optimized_SE"]].rename(
    columns={"Default_SE": "Default_AUROC_SE", "Optimized_SE": "Optimized_AUROC_SE"}
)
merged = df_compare.merge(auroc_df, on="Detector", how="inner")


# helper to compute pearson r and fit linear regression
def corr_and_fit(x, y):
    # drop NaNs
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return (np.nan, np.nan, None)
    r, p = stats.pearsonr(x[mask], y[mask])
    coef = np.polyfit(x[mask], y[mask], 1)
    fit_fn = np.poly1d(coef)
    return r, p, fit_fn


# Default
x_def = merged["Default_Accuracy"].to_numpy()
y_def = merged["Default_AUROC"].to_numpy()
r_def, p_def, fit_def = corr_and_fit(x_def, y_def)

# Optimized
x_opt = merged["Optimized_Accuracy"].to_numpy()
y_opt = merged["Optimized_AUROC"].to_numpy()
r_opt, p_opt, fit_opt = corr_and_fit(x_opt, y_opt)

# Combined
x_comb = np.concatenate([x_def, x_opt])
y_comb = np.concatenate([y_def, y_opt])
r_comb, p_comb, fit_comb = corr_and_fit(x_comb, y_comb)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
# errorbars: x = accuracy ± Accuracy_SE, y = auroc ± AUROC_SE
ax.errorbar(merged["Default_Accuracy"], merged["Default_AUROC"],
            xerr=merged.get("Default_SE"), yerr=merged.get("Default_AUROC_SE"),
            fmt='o', label="Default", alpha=0.8, capsize=3)
ax.errorbar(merged["Optimized_Accuracy"], merged["Optimized_AUROC"],
            xerr=merged.get("Optimized_SE"), yerr=merged.get("Optimized_AUROC_SE"),
            fmt='s', label="Optimized", alpha=0.9, capsize=3)

# regression lines
xs = np.linspace(
    np.nanmin([x_comb.min(), y_comb.min()]) if np.isfinite(x_comb).any() else 0,
    np.nanmax([x_comb.max(), y_comb.max()]) if np.isfinite(x_comb).any() else 1,
    100
)
if fit_def is not None:
    ax.plot(xs, fit_def(xs), color='C0', linestyle='--', alpha=0.6)
if fit_opt is not None:
    ax.plot(xs, fit_opt(xs), color='C1', linestyle='--', alpha=0.6)
if fit_comb is not None:
    ax.plot(xs, fit_comb(xs), color='grey', linestyle=':', alpha=0.6, label='Combined fit')

ax.set_xlabel("Accuracy")
ax.set_ylabel("AUROC")
ax.set_title("Correlation: Accuracy vs AUROC")
ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.grid(True, linestyle='--', alpha=0.4)

# annotate correlations
txt = (
    f"Default: r={r_def:.3f}, p={p_def:.3g}\n"
    f"Optimized: r={r_opt:.3f}, p={p_opt:.3g}\n"
    f"Combined: r={r_comb:.3f}, p={p_comb:.3g}"
)
ax.text(0.02, 0.98, txt, transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax.legend()

# save
figure_path = os.path.join(current_path, "experiment_files", "export", "fig", "comparision_unsupervised", dataset,
                           getattr(dataset_info, "transform_seq_name", "default"))
os.makedirs(figure_path, exist_ok=True)
plt.savefig(os.path.join(figure_path, "accuracy_vs_auroc.png"), dpi=200)
plt.show()

In [ ]:
import time
import os
import json
import torch
import numpy as np

# SPEED TEST
speedtest_batches = 50
speedtest_warmup_batches = 5  #
speedtest_results_dir = os.path.join(base_results_dir, "speedtests_new")
os.makedirs(speedtest_results_dir, exist_ok=True)

print("\n=== Running speed tests for detectors (fixed) ===")
for detector in detectors:
    if detector == "laplace_entropy_gridsearch":
        print(f"\n[SpeedTest] Skipping detector {detector} (too expensive).")
        continue

    print(f"\n[SpeedTest] Detector: {detector}")
    detector_dir = os.path.join(base_results_dir, detector)
    speedtest_path = os.path.join(speedtest_results_dir, f"{detector}_speed.json")

    if os.path.exists(speedtest_path):
        print(f"[SpeedTest] Found cached result at {speedtest_path}, skipping measurement.")
        continue

    # Pick params (optimized if available)
    params_path = os.path.join(detector_dir, "best_params.json")
    if os.path.exists(params_path):
        with open(params_path, "r") as f:
            best_params = json.load(f)
        print(f"[SpeedTest] Using optimized params.")
    else:
        best_params = get_default_ood_params(detector)
        print(f"[SpeedTest] Using default params.")

    # Try loading any saved model states
    best_model_path_prefix = os.path.join(detector_dir, "best_model")
    loaded_model_params = []
    idx = 0
    while os.path.exists(f"{best_model_path_prefix}_{idx}.pt"):
        loaded_model_params.append(
            torch.load(f"{best_model_path_prefix}_{idx}.pt", map_location=device)
        )
        idx += 1

    # Build OOD problem
    final_kwargs = {
        "model": model,
        "train_cache": train_cache,
        "transform_seq": transform_seq_arg,
        "dataset_info": dataset_info,
        "architecture": architecture,
        "device": str(device),
        "val_id_loader": val_loader_preshuffled_in_dist,
        "val_ood_loader": val_loader_transformed_preshuffled,
    }
    if loaded_model_params:
        final_kwargs["model_params"] = loaded_model_params

    problem = create_ood_problem(detector, best_params, **final_kwargs)

    model.eval().to(device)
    torch.cuda.empty_cache()

    test_iter = iter(test_loader_transformed)

    times = []
    measured_samples = 0

    with torch.no_grad():
        for batch_idx in range(speedtest_warmup_batches + speedtest_batches):
            try:
                data, _ = next(test_iter)
            except StopIteration:
                break

            data = data.to(device)

            # Synchronize before timing
            if torch.cuda.is_available():
                torch.cuda.synchronize(device)

            start_time = time.perf_counter()

            # Simulate detector inference
            res = optimizer_search_eval.optimize(problem, data, y=None)
            best_param = res[0] if isinstance(res, tuple) and len(res) >= 1 else res
            x_trans = problem.transform(data, best_param)
            _ = model(x_trans)

            # Synchronize after timing
            if torch.cuda.is_available():
                torch.cuda.synchronize(device)

            elapsed = time.perf_counter() - start_time

            if batch_idx < speedtest_warmup_batches:
                print(f"[SpeedTest] Warmup batch {batch_idx + 1}: {elapsed:.4f}s (ignored)")
            else:
                times.append(elapsed)
                measured_samples += data.size(0)
                print(f"[SpeedTest] Measured batch {batch_idx + 1 - speedtest_warmup_batches}: {elapsed:.4f}s")

    if len(times) == 0:
        print("[SpeedTest] No batches measured. Skipping.")
        continue

    # Summarize
    avg_batch_time = np.mean(times)
    avg_sample_time = avg_batch_time / (measured_samples / len(times))

    result = {
        "detector": detector,
        "warmup_batches": speedtest_warmup_batches,
        "batches_tested": len(times),
        "avg_time_per_batch_sec": float(avg_batch_time),
        "avg_time_per_sample_sec": float(avg_sample_time),
        "device": str(device),
        "batch_size": dataset_info.batch_size,
    }

    with open(speedtest_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"[SpeedTest] Saved results to {speedtest_path}")
    print(f"[SpeedTest] Avg: {avg_batch_time:.4f}s/batch ({avg_sample_time * 1e3:.3f} ms/sample)")

print("\nAll speed tests completed successfully.")

In [ ]:
import math
# TORCH PROFILER FLOP and memory test.


import pandas as pd
import torch
import os
import json
from torch.profiler import profile, record_function, ProfilerActivity

device_runtime = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device_runtime}")

model = model.to(device_runtime)
model.eval()

data, _ = next(iter(test_loader_transformed))
data = data.to(device_runtime)
input_size = tuple(data.shape)
batch_size = data.shape[0]

profiler_cache_dir = os.path.join(base_results_dir, "profiler_cache2")
os.makedirs(profiler_cache_dir, exist_ok=True)

results = []


def safe_div(value, divisor, ndigits):
    try:
        if math.isnan(value):
            return float("nan")
    except TypeError:
        return float("nan")
    return round(value / divisor, ndigits)


def recompute_per_batch(record, batch_size):
    record["Batch Size"] = batch_size
    record["FLOPs (GFLOPs) [profiler] / batch"] = safe_div(
        record["FLOPs (GFLOPs) [profiler]"], batch_size, 6
    )
    record["Total Peak VRAM (MB) [measured] / batch"] = safe_div(
        record["Total Peak VRAM (MB) [measured]"], batch_size, 4
    )
    record["Forward Activations (MB) [measured] / batch"] = safe_div(
        record["Forward Activations (MB) [measured]"], batch_size, 4
    )
    return record


def measure_peak_activation_mb(model_obj, data):
    """
    Measures actual peak memory delta during forward pass.
    More accurate than torchinfo for models with multiple/stochastic forward passes.
    """
    if not torch.cuda.is_available():
        return float("nan")

    model_obj.eval()
    with torch.no_grad():
        _ = model_obj(data)  # warmup

    torch.cuda.synchronize(device_runtime)
    torch.cuda.reset_peak_memory_stats(device_runtime)
    baseline_mem = torch.cuda.memory_allocated(device_runtime)

    with torch.no_grad():
        _ = model_obj(data)

    torch.cuda.synchronize(device_runtime)
    peak_mem = torch.cuda.max_memory_allocated(device_runtime)

    activation_mb = (peak_mem - baseline_mem) / (1024 ** 2)
    return round(max(activation_mb, 0.0), 2)


def extract_summary_stats_profiler(model_obj, name, cache_path, batch_size):
    if os.path.exists(cache_path):
        with open(cache_path, "r") as f:
            cached = json.load(f)
        if cached.get("Batch Size") != batch_size:
            print(f"  [profiler] Batch size changed for {name}, recomputing per-batch values.")
            cached = recompute_per_batch(cached, batch_size)
            with open(cache_path, "w") as f:
                json.dump(cached, f, indent=2)
        else:
            print(f"  [profiler] Loaded cached results for {name}.")
        return cached

    model_obj = model_obj.to(device_runtime)
    model_obj.eval()

    total_params = sum(p.numel() for p in model_obj.parameters())
    trainable_params = sum(p.numel() for p in model_obj.parameters() if p.requires_grad)
    non_trainable_params = total_params - trainable_params
    buffer_params = sum(b.numel() for b in model_obj.buffers())

    param_size_mb = total_params * 4 / (1024 ** 2)
    buffer_size_mb = buffer_params * 4 / (1024 ** 2)

    activation_only_mb = measure_peak_activation_mb(model_obj, data)
    total_peak_mb = round(param_size_mb + buffer_size_mb + activation_only_mb, 2)

    # FLOPs via torch.profiler
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    try:
        with torch.no_grad():
            _ = model_obj(data)  # warmup

        with profile(
                activities=activities,
                with_flops=True,
                record_shapes=True,
        ) as prof:
            with record_function("model_inference"):
                with torch.no_grad():
                    _ = model_obj(data)

        total_flops = sum(e.flops for e in prof.key_averages() if e.flops > 0)
        flops_gflops = round(total_flops / 1e9, 4)

        cpu_mem_mb = round(
            sum(e.cpu_memory_usage for e in prof.key_averages() if e.cpu_memory_usage > 0)
            / (1024 ** 2), 2
        )

    except Exception as e:
        print(f"  [profiler] Failed for {name}: {e}")
        flops_gflops = float("nan")
        cpu_mem_mb = float("nan")

    result = {
        "Component": name,
        "Batch Size": batch_size,
        "Params (M)": round(total_params / 1e6, 3),
        "Trainable (M)": round(trainable_params / 1e6, 3),
        "Non-Trainable (M)": round(non_trainable_params / 1e6, 3),
        "Buffers (M)": round(buffer_params / 1e6, 3),
        "Param Size (MB)": round(param_size_mb, 2),
        "Buffer Size (MB)": round(buffer_size_mb, 2),
        "Forward Activations (MB) [measured]": activation_only_mb,
        "Forward Activations (MB) [measured] / batch": safe_div(activation_only_mb, batch_size, 4),
        "Total Peak VRAM (MB) [measured]": total_peak_mb,
        "Total Peak VRAM (MB) [measured] / batch": safe_div(total_peak_mb, batch_size, 4),
        "FLOPs (GFLOPs) [profiler]": flops_gflops,
        "FLOPs (GFLOPs) [profiler] / batch": safe_div(flops_gflops, batch_size, 6),
        "CPU Mem (MB) [profiler]": cpu_mem_mb,
    }

    with open(cache_path, "w") as f:
        json.dump(result, f, indent=2)
    print(f"  [profiler] Saved results for {name} to {cache_path}.")

    return result


# Main Model
main_model_cache_path = os.path.join(profiler_cache_dir, "main_model.json")
results.append(extract_summary_stats_profiler(model, "Main Model", main_model_cache_path, batch_size))

# Detectors
for detector_name in detectors:
    if detector_name == "laplace_entropy_gridsearch":
        continue

    print(f"\n[profiler] Detector: {detector_name}")
    detector_dir = os.path.join(base_results_dir, detector_name)
    detector_cache_path = os.path.join(profiler_cache_dir, f"{detector_name}.json")

    if os.path.exists(detector_cache_path):
        with open(detector_cache_path, "r") as f:
            cached = json.load(f)
        if cached.get("Batch Size") != batch_size:
            print(f"  [profiler] Batch size changed for {detector_name}, recomputing per-batch values.")
            cached = recompute_per_batch(cached, batch_size)
            with open(detector_cache_path, "w") as f:
                json.dump(cached, f, indent=2)
        else:
            print(f"  [profiler] Loaded cached results for {detector_name}.")
        results.append(cached)
        continue

    params_path = os.path.join(detector_dir, "best_params.json")
    if os.path.exists(params_path):
        with open(params_path, "r") as f:
            best_params = json.load(f)
    else:
        best_params = get_default_ood_params(detector_name)

    final_kwargs = {
        "model": model,
        "train_cache": train_cache,
        "transform_seq": transform_seq_arg,
        "dataset_info": dataset_info,
        "architecture": architecture,
        "device": device_runtime,
        "val_id_loader": val_loader_preshuffled_in_dist,
        "val_ood_loader": val_loader_transformed_preshuffled,
    }

    best_model_path_prefix = os.path.join(detector_dir, "best_model")
    loaded_model_params = []
    idx = 0
    while os.path.exists(f"{best_model_path_prefix}_{idx}.pt"):
        loaded_model_params.append(
            torch.load(f"{best_model_path_prefix}_{idx}.pt", map_location=device_runtime)
        )
        idx += 1
    if loaded_model_params:
        final_kwargs["model_params"] = loaded_model_params

    detector = create_ood_problem(detector_name, best_params, **final_kwargs)
    detector = detector.confidence_module.to(device_runtime)
    detector.eval()

    results.append(extract_summary_stats_profiler(detector, detector_name, detector_cache_path, batch_size))

df = pd.DataFrame(results)
df

In [ ]:
detectors_all = ["knn", "per_class_knn", "knn_mixed", "per_class_knn_mixed", "vim", "dice", "she", "laplace_mi",
                 "laplace_weighted", "trust_score", "openmax", "mahalanobis", "rmd", "class_prototype", "energy",
                 "per_class_prototype", "laplace_entropy", "adjusted_entropy", "entropy", "MSP", "MaxLogit"]
#include ash last layer and react
detectors_all_images = detectors_all + ["ash", "react_all"]
detectors_all_non_images = detectors_all + ["ash_last", "react"]

detectors_com = detectors_all + ["ash", "react_all", "ash_last", "react"]


In [ ]:
dataset_info

In [ ]:
import os
import json
import numpy as np
import pandas as pd


print("\n=== Creating LaTeX table with relative times ===")

datasets_to_plot = ["bigger_mnist", "bigger_emnist", "tu_berlin", "modelnet10"]
dataset_labels = ["Big MNIST", "Big EMNIST", "TU Berlin", "ModelNet10"]

# Collect speed results for all datasets
all_speed_data = {}
for ds in datasets_to_plot:
    ds_info = get_dataset_info(ds)
    ds_transform_name = getattr(ds_info, "transform_seq_name", "default")

    speed_dir = os.path.join(
        current_path,
        "experiment_files",
        "results",
        "comparison_unsupervised",
        ds,
        default_architecutre_mapping[ds],
        ds_transform_name,
        "speedtests"
    )

    speed_results = []
    ds_list = detectors_all_images if ds_info.datatype == "image" else detectors_all_non_images
    for detector in ds_list:
        speedtest_path = os.path.join(speed_dir, f"{detector}_speed.json")
        if os.path.exists(speedtest_path):
            with open(speedtest_path, "r") as f:
                data = json.load(f)
                speed_results.append(data)

    all_speed_data[ds] = speed_results

# Build DataFrame with relative times
rows = []
for ds, ds_label in zip(datasets_to_plot, dataset_labels):
    # Find energy baseline time for this dataset
    energy_time = next(
        (r["avg_time_per_sample_sec"] * 1000
         for r in all_speed_data[ds] if r["detector"] == "energy"),
        None
    )

    if energy_time is None:
        print(f"Warning: No energy baseline found for {ds}")
        continue

    ds_info = get_dataset_info(ds)
    ds_transform_name = getattr(ds_info, "transform_seq_name", "default")

    for detector in (detectors_all_images if ds_info.datatype == "image" else detectors_all_non_images):
        det_time = next(
            (r["avg_time_per_sample_sec"] * 1000
             for r in all_speed_data[ds] if r["detector"] == detector),
            None
        )

        if det_time is not None:
            relative_time = det_time / energy_time
            rows.append({
                "Detector": name_map.get(detector, detector),
                "Dataset": ds_label,
                "Relative Time": relative_time
            })

df = pd.DataFrame(rows)

# Pivot to get detectors as rows, datasets as columns (maintaining order)
table = df.pivot(index="Detector", columns="Dataset", values="Relative Time")
# Reorder columns to match dataset_labels order
table = table[dataset_labels]

# Sort by average relative time
table = table.sort_index()

# Format as LaTeX
latex_str = table.to_latex(
    float_format="%.2f",
    caption="Relative inference time compared to Energy baseline (Energy = 1.0)",
    label="tab:speed_comparison",
    escape=False
)

# Save to file
latex_path = os.path.join(base_results_dir, "..", "speed_comparison_relative.tex")
with open(latex_path, "w") as f:
    f.write(latex_str)

print(f" Saved LaTeX table to {latex_path}")
print("\n" + latex_str)

In [ ]:
#for new results with new gpu
import os
import json
import numpy as np
import pandas as pd


print("\n=== Creating LaTeX table with relative times ===")

datasets_to_plot = ["bigger_mnist", "bigger_emnist", "tu_berlin", "modelnet10"]
dataset_labels = ["Big MNIST", "Big EMNIST", "TU Berlin", "ModelNet10"]

# Collect speed results for all datasets
all_speed_data = {}
for ds in datasets_to_plot:
    ds_info = get_dataset_info(ds)
    ds_transform_name = getattr(ds_info, "transform_seq_name", "default")

    speed_dir = os.path.join(
        current_path,
        "experiment_files",
        "results",
        "comparison_unsupervised",
        ds,
        default_architecutre_mapping[ds],
        ds_transform_name,
        "speedtests_new"
    )

    speed_results = []
    ds_list = detectors_all_images if ds_info.datatype == "image" else detectors_all_non_images
    for detector in ds_list:
        speedtest_path = os.path.join(speed_dir, f"{detector}_speed.json")
        if os.path.exists(speedtest_path):
            with open(speedtest_path, "r") as f:
                data = json.load(f)
                speed_results.append(data)

    all_speed_data[ds] = speed_results

# Build DataFrame with relative times
rows = []
baseline_times = {}  # Store absolute baseline times for the final row

for ds, ds_label in zip(datasets_to_plot, dataset_labels):
    # Find energy baseline time for this dataset (in ms)
    energy_time = next(
        (r["avg_time_per_sample_sec"] * 1000
         for r in all_speed_data[ds] if r["detector"] == "energy"),
        None
    )

    if energy_time is None:
        print(f"Warning: No energy baseline found for {ds}")
        continue

    # Store for the footer
    baseline_times[ds_label] = energy_time

    ds_info = get_dataset_info(ds)
    for detector in (detectors_all_images if ds_info.datatype == "image" else detectors_all_non_images):
        det_time = next(
            (r["avg_time_per_sample_sec"] * 1000
             for r in all_speed_data[ds] if r["detector"] == detector),
            None
        )

        if det_time is not None:
            # Relative time: 1.0 means same as energy
            relative_time = det_time / energy_time
            rows.append({
                "Detector": name_map.get(detector, detector),
                "Dataset": ds_label,
                "Relative Time": relative_time
            })

df = pd.DataFrame(rows)

# Pivot to get detectors as rows, datasets as columns
table = df.pivot(index="Detector", columns="Dataset", values="Relative Time")
table = table[dataset_labels]
table = table.sort_index()

# Add the Baseline Absolute Time row at the bottom
baseline_row = pd.Series(baseline_times, name="Baseline (ms)")
table = pd.concat([table, baseline_row.to_frame().T])

# Format as LaTeX
latex_str = table.to_latex(
    float_format="%.2f",
    caption="Relative inference time compared to Energy baseline (Energy = 1.00). Bottom row shows absolute Energy time in ms.",
    label="tab:speed_comparison",
    escape=False,
    column_format="l" + "c" * len(dataset_labels)  # Left align detector, center datasets
)

# Save to file
latex_path = os.path.join(base_results_dir, "..", "speed_comparison_relative.tex")
with open(latex_path, "w") as f:
    f.write(latex_str)

print(f" Saved LaTeX table to {latex_path}")
print("\n" + latex_str)

In [ ]:
import os
import json
import pandas as pd



datasets_to_plot = ["bigger_mnist", "bigger_emnist", "tu_berlin", "modelnet10"]
dataset_labels = ["MNIST", "EMNIST", "TU Berlin", "ModelNet10"]

metric_map = {
    "Size (MB)": "Sz",
    "VRAM (MB)": "VR",
    "GFLOPs": "GF"
}


# Helper for 1 decimal place rounding
def format_fixed_decimal(val):
    try:
        if isinstance(val, (int, float)):
            return f"{float(val):.1f}"
        return str(val)
    except:
        return str(val)


rows = []

for ds, ds_label in zip(datasets_to_plot, dataset_labels):
    ds_info = get_dataset_info(ds)
    ds_transform_name = getattr(ds_info, "transform_seq_name", "default")
    architecture2 = default_architecutre_mapping[ds]

    profiler_cache_dir = os.path.join(
        current_path, "experiment_files", "results", "comparison_unsupervised",
        ds, architecture2, ds_transform_name, "profiler_cache2"
    )

    ds_list = detectors_all_images if ds_info.datatype == "image" else detectors_all_non_images

    # Process Main Model and Detectors
    components = [("main_model", "Main Model")] + [(d, name_map.get(d, d)) for d in ds_list if
                                                   d != "laplace_entropy_gridsearch"]

    for file_id, clean_name in components:
        cache_path = os.path.join(profiler_cache_dir, f"{file_id}.json")

        if os.path.exists(cache_path):
            with open(cache_path, "r") as f:
                r = json.load(f)

            sz_val = r.get("Param Size (MB)", 0) + r.get("Buffer Size (MB)", 0)
            vr_val = r.get("Forward Activations (MB) [measured]", 0)
            gf_val = r.get("FLOPs (GFLOPs) [profiler]", 0)

            rows.append({
                "Dataset": ds_label,
                "Component": clean_name,
                metric_map["Size (MB)"]: format_fixed_decimal(sz_val),
                metric_map["VRAM (MB)"]: format_fixed_decimal(vr_val),
                metric_map["GFLOPs"]: format_fixed_decimal(gf_val),
            })

df_all = pd.DataFrame(rows)

# Handle TU Berlin GFLOPs special case
df_all.loc[df_all["Dataset"] == "TU Berlin", metric_map["GFLOPs"]] = "-"

# Pivot: Group by Dataset (top level) and Metric (second level)
table = df_all.pivot(index="Component", columns="Dataset", values=list(metric_map.values()))
table = table.reorder_levels(["Dataset", None], axis=1)
table = table[dataset_labels]  # Keep dataset order
table = table.sort_index()

num_data_cols = len(datasets_to_plot) * len(metric_map)

col_spec = "l" + "*{%d}{wc{0.9cm}}" % num_data_cols

latex_main = table.to_latex(
    caption=f"Profiler summary ({metric_map['Size (MB)']}: Size MB, {metric_map['VRAM (MB)']}: VRAM MB, {metric_map['GFLOPs']}: GFLOPs).",
    label="tab:profiler_summary",
    escape=False,
    multicolumn=True,
    multicolumn_format="c",
    column_format=col_spec
)

# Wrapping in resizebox to ensure it stays within page margins
latex_wrapped = (

    f"{latex_main}"
)

latex_path = os.path.join(
    current_path, "experiment_files", "results", "comparison_unsupervised", "profiler_summary.tex"
)

with open(latex_path, "w") as f:
    f.write(latex_wrapped)

print(f"Saved to {latex_path}")
print("\n" + latex_wrapped)

table

In [ ]:
import os
import json
import numpy as np
import pandas as pd


print("\n=== Creating LaTeX table with relative times ===")

datasets_to_plot = ["bigger_mnist", "bigger_emnist", "tu_berlin", "modelnet10"]
dataset_labels = ["Big MNIST", "Big EMNIST", "TU Berlin", "ModelNet10"]

# Collect speed results for all datasets
all_speed_data = {}
for ds in datasets_to_plot:
    ds_info = get_dataset_info(ds)
    ds_transform_name = getattr(ds_info, "transform_seq_name", "default")

    speed_dir = os.path.join(
        current_path,
        "experiment_files",
        "results",
        "comparison_unsupervised",
        ds,
        default_architecutre_mapping[ds],
        ds_transform_name,
        "speedtests"
    )

    speed_results = []
    ds_list = detectors_all_images if ds_info.datatype == "image" else detectors_all_non_images
    for detector in ds_list:
        speedtest_path = os.path.join(speed_dir, f"{detector}_speed2.json")
        if os.path.exists(speedtest_path):
            with open(speedtest_path, "r") as f:
                data = json.load(f)
                speed_results.append(data)

    all_speed_data[ds] = speed_results

# Build DataFrame with relative times
rows = []
for ds, ds_label in zip(datasets_to_plot, dataset_labels):
    # Find energy baseline time for this dataset
    energy_time = next(
        (r["avg_time_per_sample_sec"] * 1000
         for r in all_speed_data[ds] if r["detector"] == "energy"),
        None
    )

    if energy_time is None:
        print(f"Warning: No energy baseline found for {ds}")
        continue

    ds_info = get_dataset_info(ds)
    ds_transform_name = getattr(ds_info, "transform_seq_name", "default")

    for detector in (detectors_all_images if ds_info.datatype == "image" else detectors_all_non_images):
        det_time = next(
            (r["avg_time_per_sample_sec"] * 1000
             for r in all_speed_data[ds] if r["detector"] == detector),
            None
        )

        if det_time is not None:
            relative_time = det_time / energy_time
            rows.append({
                "Detector": name_map.get(detector, detector),
                "Dataset": ds_label,
                "Relative Time": relative_time
            })

df = pd.DataFrame(rows)

# Pivot to get detectors as rows, datasets as columns (maintaining order)
table = df.pivot(index="Detector", columns="Dataset", values="Relative Time")
# Reorder columns to match dataset_labels order
table = table[dataset_labels]

# Sort by average relative time
table = table.sort_index()

# Format as LaTeX
latex_str = table.to_latex(
    float_format="%.2f",
    caption="Relative inference time compared to Energy baseline (Energy = 1.0)",
    label="tab:speed_comparison",
    escape=False
)

# Save to file
latex_path = os.path.join(base_results_dir, "..", "speed_comparison_relative.tex")
with open(latex_path, "w") as f:
    f.write(latex_str)

print(f" Saved LaTeX table to {latex_path}")
print("\n" + latex_str)

In [ ]:
import os
import json
import numpy as np
import pandas as pd


datasets_to_compare = ["bigger_mnist", "bigger_emnist", "tu_berlin", "modelnet10"]
dataset_labels = {
    "bigger_mnist": "MNIST",
    "bigger_emnist": "EMNIST",
    "tu_berlin": "TU Berlin",
    "modelnet10": "ModelNet10",
}

default_arch = default_architecutre_mapping


def _safe_load_json(path: str):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def _extract_acc(res):
    """
    Returns (mean, se) from a saved eval json dict.
    """
    if not isinstance(res, dict):
        return (np.nan, np.nan)

    # direct keys
    mean = res.get("accuracy_mean", res.get("accuracy", np.nan))
    se = res.get("accuracy_se", np.nan)

    # if per_run exists, compute mean/se if missing
    if (not np.isfinite(mean) or not np.isfinite(se)) and isinstance(res.get("per_run"), list):
        vals = []
        for r in res["per_run"]:
            if isinstance(r, dict) and "accuracy" in r:
                vals.append(r["accuracy"])
            elif isinstance(r, (float, int)):
                vals.append(r)
        if vals:
            a = np.asarray(vals, dtype=float)
            mean = float(np.nanmean(a))
            se = float(np.nanstd(a) / np.sqrt(np.sum(np.isfinite(a))))

    try:
        mean = float(mean)
    except Exception:
        mean = np.nan
    try:
        se = float(se)
    except Exception:
        se = np.nan

    return (mean, se)


def _results_dir_for_dataset(ds: str) -> str:
    ds_info = get_dataset_info(ds)
    ds_transform_name = getattr(ds_info, "transform_seq_name", "default")
    arch = default_arch[ds]
    return os.path.join(
        current_path,
        "experiment_files",
        "results",
        "comparison_unsupervised",
        ds,
        arch,
        ds_transform_name,
    )


rows = []
for ds in datasets_to_compare:
    result_dir = _results_dir_for_dataset(ds)
    det_list = detectors_com

    for det in det_list:
        p_tuned = os.path.join(result_dir, det, "eval_results.json")
        p_def = os.path.join(result_dir, det, "eval_results_default.json")

        res_tuned = _safe_load_json(p_tuned) if os.path.exists(p_tuned) else None
        res_def = _safe_load_json(p_def) if os.path.exists(p_def) else None

        tuned_mean, tuned_se = _extract_acc(res_tuned)
        def_mean, def_se = _extract_acc(res_def)

        rows.append({
            "dataset": ds,
            "dataset_label": dataset_labels.get(ds, ds),
            "detector": det,
            "detector_label": name_map.get(det, det) if "name_map" in globals() else det,
            "default_acc": def_mean,
            "default_se": def_se,
            "tuned_acc": tuned_mean,
            "tuned_se": tuned_se,
        })

df_all = pd.DataFrame(rows)

table_wide = df_all.pivot_table(
    index="detector_label",
    columns="dataset_label",
    values=["default_acc", "tuned_acc"],
    aggfunc="first",
)

table_wide.columns = [f"{ds} | {metric}" for metric, ds in table_wide.columns.to_list()]
table_wide = table_wide.sort_index()

#Best-of summary per dataset
df_all["best_acc"] = df_all[["default_acc", "tuned_acc"]].max(axis=1)
best_table = df_all.pivot_table(
    index="detector_label",
    columns="dataset_label",
    values="best_acc",
    aggfunc="first",
).sort_index()

#Add averages across datasets
table_wide["Mean | default_acc"] = df_all.groupby("detector_label")["default_acc"].mean()
table_wide["Mean | tuned_acc"] = df_all.groupby("detector_label")["tuned_acc"].mean()
best_table["Mean"] = df_all.groupby("detector_label")["best_acc"].mean()

out_dir = os.path.join(current_path, "experiment_files", "export", "tables")
os.makedirs(out_dir, exist_ok=True)

wide_csv = os.path.join(out_dir, "accuracy_comparison_default_vs_tuned.csv")
best_csv = os.path.join(out_dir, "accuracy_comparison_best.csv")
table_wide.to_csv(wide_csv)
best_table.to_csv(best_csv)

# Optional LaTeX exports
wide_tex = os.path.join(out_dir, "accuracy_comparison_default_vs_tuned.tex")
best_tex = os.path.join(out_dir, "accuracy_comparison_best.tex")
table_wide.to_latex(wide_tex, float_format="%.4f", escape=False)
best_table.to_latex(best_tex, float_format="%.4f", escape=False)

print(f"Saved: {wide_csv}")
print(f"Saved: {best_csv}")
print(f"Saved: {wide_tex}")
print(f"Saved: {best_tex}")

# Display in-notebook
display(table_wide)
display(best_table)

In [ ]:


def _fmt_cell(def_acc, def_se, opt_acc, opt_se):
    if not np.isfinite(def_acc) and not np.isfinite(opt_acc):
        return ""
    def_str = f"{def_acc:.3f} $\\pm$ {def_se:.3f}"
    opt_str = f"{opt_acc:.3f} $\\pm$ {opt_se:.3f}"
    return f"\\makecell{{{def_str} \\\\ {opt_str}}}"


table_stacked = df_all.pivot_table(
    index="detector_label",
    columns="dataset_label",
    values=["default_acc", "default_se", "tuned_acc", "tuned_se"],
    aggfunc="first",
)

stacked = pd.DataFrame(index=table_stacked.index)

for ds in table_stacked.columns.levels[1]:
    d_acc = table_stacked[("default_acc", ds)]
    d_se = table_stacked[("default_se", ds)]
    t_acc = table_stacked[("tuned_acc", ds)]
    t_se = table_stacked[("tuned_se", ds)]

    stacked[ds] = [
        _fmt_cell(da, ds_, ta, ts)
        for da, ds_, ta, ts in zip(d_acc, d_se, t_acc, t_se)
    ]

stacked = stacked.sort_index()


out_dir = os.path.join(current_path, "experiment_files", "export", "tables")
os.makedirs(out_dir, exist_ok=True)

stacked_tex = os.path.join(out_dir, "accuracy_default_over_optimized_with_uncertainty.tex")
print(stacked.to_latex(
    stacked_tex,
    escape=False,
    column_format="l" + "c" * len(stacked.columns),
))

print(f"Saved compact LaTeX table with uncertainty: {stacked_tex}")

display(stacked)


In [ ]:
from src.utils.eval.vis import load_json, extract_per_run_metrics, summarize_runs, choose_accuracy_metric, \
    plot_group_short, \
    build_and_write_latex, plt_setup_paper

W = plt_setup_latex(W=5.53)
figure_path2 = os.path.join(current_path, "experiment_files", "export", "fig3", "comparision_unsupervised", dataset,
                            transform_name)
os.makedirs(figure_path2, exist_ok=True)

In [ ]:
detectors_all = ["knn_mixed", "per_class_knn_mixed", "knn", "per_class_knn", "vim", "dice", "she", "laplace_mi",
                 "laplace_weighted", "trust_score", "openmax", "mahalanobis", "rmd", "class_prototype", "energy",
                 "per_class_prototype", "laplace_entropy", "adjusted_entropy", "entropy", "MSP", "MaxLogit"]
#include ash last layer and react
detectors_all_images = detectors_all + ["ash", "react_all"]
detectors_all_non_images = detectors_all + ["ash_last", "react"]

detectors_com = detectors_all + ["ash", "react_all", "ash_last", "react"]


In [ ]:
import os
import json
import numpy as np
import pandas as pd


datasets_to_compare = ["bigger_mnist", "bigger_emnist", "tu_berlin", "modelnet10"]
dataset_labels = {
    "bigger_mnist": "MNIST",
    "bigger_emnist": "EMNIST",
    "tu_berlin": "TU Berlin",
    "modelnet10": "ModelNet10",
}

default_arch = default_architecutre_mapping


def _safe_load_json(path: str):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def _extract_acc(res):
    """
    Returns (mean, se) from a saved eval json dict.
    """
    if not isinstance(res, dict):
        return (np.nan, np.nan)

    # direct keys
    mean = res.get("accuracy_mean", res.get("accuracy", np.nan))
    se = res.get("accuracy_se", np.nan)

    # if per_run exists, compute mean/se if missing
    if (not np.isfinite(mean) or not np.isfinite(se)) and isinstance(res.get("per_run"), list):
        vals = []
        for r in res["per_run"]:
            if isinstance(r, dict) and "accuracy" in r:
                vals.append(r["accuracy"])
            elif isinstance(r, (float, int)):
                vals.append(r)
        if vals:
            a = np.asarray(vals, dtype=float)
            mean = float(np.nanmean(a))
            se = float(np.nanstd(a) / np.sqrt(np.sum(np.isfinite(a))))

    try:
        mean = float(mean)
    except Exception:
        mean = np.nan
    try:
        se = float(se)
    except Exception:
        se = np.nan

    return (mean, se)


def _results_dir_for_dataset(ds: str) -> str:
    ds_info = get_dataset_info(ds)
    ds_transform_name = getattr(ds_info, "transform_seq_name", "default")
    arch = default_arch[ds]
    return os.path.join(
        current_path,
        "experiment_files",
        "results",
        "comparison_unsupervised",
        ds,
        arch,
        ds_transform_name,
    )


rows = []
for ds in datasets_to_compare:
    result_dir = _results_dir_for_dataset(ds)
    det_list = detectors_com

    for det in det_list:
        p_tuned = os.path.join(result_dir, det, "eval_results.json")
        p_def = os.path.join(result_dir, det, "eval_results_default.json")
        print(p_def)
        print(os.path.exists(p_def))
        res_tuned = _safe_load_json(p_tuned) if os.path.exists(p_tuned) else None
        res_def = _safe_load_json(p_def) if os.path.exists(p_def) else None

        tuned_mean, tuned_se = _extract_acc(res_tuned)
        def_mean, def_se = _extract_acc(res_def)

        rows.append({
            "dataset": ds,
            "dataset_label": dataset_labels.get(ds, ds),
            "detector": det,
            "detector_label": name_map.get(det, det) if "name_map" in globals() else det,
            "default_acc": def_mean,
            "default_se": def_se,
            "tuned_acc": tuned_mean,
            "tuned_se": tuned_se,
        })

df_all = pd.DataFrame(rows)

# Wide comparison table (Default vs Tuned per dataset)
table_wide = df_all.pivot_table(
    index="detector_label",
    columns="dataset_label",
    values=["default_acc", "tuned_acc"],
    aggfunc="first",
)

# Flatten MultiIndex columns -> "Dataset | default_acc"
table_wide.columns = [f"{ds} | {metric}" for metric, ds in table_wide.columns.to_list()]
table_wide = table_wide.sort_index()

#Best-of summary per dataset (max of default/tuned)
df_all["best_acc"] = df_all[["default_acc", "tuned_acc"]].max(axis=1)
best_table = df_all.pivot_table(
    index="detector_label",
    columns="dataset_label",
    values="best_acc",
    aggfunc="first",
).sort_index()

#Add averages across datasets (optional)
table_wide["Mean | default_acc"] = df_all.groupby("detector_label")["default_acc"].mean()
table_wide["Mean | tuned_acc"] = df_all.groupby("detector_label")["tuned_acc"].mean()
best_table["Mean"] = df_all.groupby("detector_label")["best_acc"].mean()

out_dir = os.path.join(current_path, "experiment_files", "export", "tables")
os.makedirs(out_dir, exist_ok=True)

wide_csv = os.path.join(out_dir, "accuracy_comparison_default_vs_tuned.csv")
best_csv = os.path.join(out_dir, "accuracy_comparison_best.csv")
table_wide.to_csv(wide_csv)
best_table.to_csv(best_csv)

# Optional LaTeX exports
wide_tex = os.path.join(out_dir, "accuracy_comparison_default_vs_tuned.tex")
best_tex = os.path.join(out_dir, "accuracy_comparison_best.tex")
table_wide.to_latex(wide_tex, float_format="%.4f", escape=False)
best_table.to_latex(best_tex, float_format="%.4f", escape=False)

print(f"Saved: {wide_csv}")
print(f"Saved: {best_csv}")
print(f"Saved: {wide_tex}")
print(f"Saved: {best_tex}")

# Display in-notebook
display(table_wide)
display(best_table)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# Identify all unique datasets in the aggregated data
all_datasets = df_all["dataset"].unique()

for ds in all_datasets:
    # 1. Filter and preprocess data for this specific dataset
    df_ds = df_all[df_all["dataset"] == ds].copy()
    df_ds["display_name"] = df_ds["detector"].replace(name_map)

    # Merge variants (e.g., ash/ash_last)
    df_ds = df_ds.sort_values("tuned_acc", ascending=False).drop_duplicates("display_name")

    # 2. Sort by performance for the Y-axis (ascending=True places best at the top in barh)
    df_ds["max_acc"] = df_ds[["default_acc", "tuned_acc"]].max(axis=1)
    df_ds = df_ds.sort_values("max_acc", ascending=True)

    # 3. Setup Plot
    y = np.arange(len(df_ds))
    height = 0.35
    fig, ax = plt.subplots(figsize=(W / 2.0, W * 0.55))

    def_err = df_ds["default_se"] * 1.96
    opt_err = df_ds["tuned_se"] * 1.96

    ax.barh(y - height / 2, df_ds["default_acc"], height, xerr=def_err,
            label="Default", color="C0", alpha=0.8, capsize=0.8,
            error_kw=dict(lw=0.5, capthick=0.5), zorder=2)

    ax.barh(y + height / 2, df_ds["tuned_acc"], height, xerr=opt_err,
            label="Tuned", color="C1", alpha=0.8, capsize=0.8,
            error_kw=dict(lw=0.5, capthick=0.5), zorder=2)

    # 4. Formatting
    ax.set_ylabel("Detector", fontsize=9)
    ax.set_xlabel("Accuracy", fontsize=9)
    ax.set_yticks(y)
    ax.set_yticklabels(df_ds["display_name"], fontsize=7)
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis="x", linestyle="--", alpha=0.6, zorder=1)

    legend_loc = "lower right" if ds == "modelnet10" else "upper left"
    ax.legend(fontsize=8, framealpha=0.4, loc=legend_loc)

    # Dynamic X-axis Zoom
    valid = df_ds["default_acc"].notna() | df_ds["tuned_acc"].notna()
    if valid.any():
        x_min = min((df_ds["default_acc"] - def_err).min(), (df_ds["tuned_acc"] - opt_err).min())
        x_max = max((df_ds["default_acc"] + def_err).max(), (df_ds["tuned_acc"] + opt_err).max())
        margin = (x_max - x_min) * 0.05
        ax.set_xlim(max(0, x_min - margin), min(1.0, x_max + margin))

    # 5. Save and Export
    # Specific transform_name for this dataset
    ds_transform = getattr(get_dataset_info(ds), "transform_seq_name", "default")
    figure_path = os.path.join(current_path, "experiment_files", "export", "fig3", "comparision_unsupervised", ds,
                               ds_transform)
    os.makedirs(figure_path, exist_ok=True)

    plt.tight_layout()
    plt.savefig(os.path.join(figure_path, "comparision_detectors.pdf"), bbox_inches="tight")
    plt.savefig(os.path.join(figure_path, "comparision_detectors.pgf"), bbox_inches="tight")
    plt.show()

    print(f"Exported individual plots for {ds} to {figure_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

SHOW_CATEGORY_LABELS = True
CAT_LABEL_X_POS = -0.10
Y_TICK_PADDING = 10
LINE_EXTENSION = -0.9

#choose one of 'star', 'background', 'thick_outline', or None
highlight_mode = 'background'

# 1. Preprocess data
df_all_plot = df_all.copy()
df_all_plot["display_name"] = df_all_plot["detector"].replace(name_map)
df_all_plot = df_all_plot.sort_values("tuned_acc", ascending=False).drop_duplicates(["dataset", "display_name"])

category_map = {
    'MSP': 'Logit', 'MaxLogit': 'Logit', 'Energy': 'Logit', 'Entropy': 'Logit',
    'OpenMax': 'Logit', 'VIM': 'Logit', 'Lapl. Entropy': 'Logit',
    'Lapl. Weighted': 'Logit', 'Lapl. MI': 'Logit', 'GEN': 'Logit',
    'Adjusted Entropy': 'Logit',
    'ReAct': 'Rect.', 'ReAct-All': 'Rect.', 'ASH': 'Rect.',
    'ASH-Last': 'Rect.', 'DICE': 'Rect.',
    'MD': 'Proto', 'RMD': 'Proto', 'SHE': 'Proto', 'Proto': 'Proto',
    'PC-Proto': 'Proto', 'Class Proto': 'Proto',
    'kNN': 'kNN', 'KNN Mix': 'kNN', 'PC-kNN Mix': 'kNN', 'Trust Score': 'kNN'
}


def get_category(name):
    if name in category_map: return category_map[name]
    name_lower = str(name).lower()
    if 'knn' in name_lower: return 'kNN'
    if 'proto' in name_lower: return 'Proto'
    return 'Other'


def get_cat_idx(name):
    cat_order = ['Logit', 'Rect.', 'Proto', 'kNN', 'Other']
    return cat_order.index(get_category(name))


# 2. Define global categorical order
unique_names = df_all_plot["display_name"].unique()
sorted_names = sorted(unique_names, key=lambda x: (get_cat_idx(x), x), reverse=True)

# 3. Setup datasets and figure layout
datasets_to_plot = ["bigger_mnist", "bigger_emnist", "tu_berlin", "modelnet10"]
n_ds = len(datasets_to_plot)
height = 0.3

# Find best detector per dataset (highest tuned_acc)
best_per_dataset = {}
for ds in datasets_to_plot:
    ds_df = df_all_plot[df_all_plot["dataset"] == ds]
    if not ds_df.empty:
        best_per_dataset[ds] = ds_df.loc[ds_df["tuned_acc"].idxmax(), "display_name"]
fig, axes = plt.subplots(1, n_ds, figsize=(W, W * 0.58), sharey=True)
plt.subplots_adjust(wspace=0.1)

for i, ds in enumerate(datasets_to_plot):
    ax = axes[i]
    ds_df = df_all_plot[df_all_plot["dataset"] == ds].set_index("display_name").reindex(sorted_names).reset_index()
    ds_df['category'] = ds_df['display_name'].apply(get_category)
    ds_df.loc[ds_df['display_name'] == 'Energy', 'tuned_acc'] = np.nan
    ds_df.loc[ds_df['display_name'] == 'Energy', 'tuned_se'] = 0

    y = np.arange(len(ds_df))
    def_err = ds_df["default_se"] * 1.96
    opt_err = ds_df["tuned_se"] * 1.96

    best_name = best_per_dataset.get(ds)
    best_y_idx = ds_df[ds_df["display_name"] == best_name].index.tolist()
    best_y = best_y_idx[0] if best_y_idx else None

    if highlight_mode == 'background' and best_y is not None:
        ax.axhspan(best_y - 0.4, best_y + 0.4, color='gold', alpha=0.35, zorder=0)

    # Plot Bars
    ax.barh(y - height / 2, ds_df["default_acc"], height, xerr=def_err,
            label="Default", color="C0", alpha=0.8, capsize=0.8,
            error_kw=dict(lw=0.5, capthick=0.5), zorder=2)

    ax.barh(y + height / 2, ds_df["tuned_acc"], height, xerr=opt_err,
            label="Tuned", color="C1", alpha=0.8, capsize=0.8,
            error_kw=dict(lw=0.5, capthick=0.5), zorder=2)

    if highlight_mode == 'thick_outline' and best_y is not None:
        best_tuned_val = ds_df.loc[best_y, "tuned_acc"]
        if pd.notna(best_tuned_val):
            ax.barh(best_y + height / 2, best_tuned_val, height,
                    color='C1', alpha=0.5, zorder=0, edgecolor='gold', linewidth=3)
        best_def_val = ds_df.loc[best_y, "default_acc"]
        if pd.notna(best_def_val):
            ax.barh(best_y - height / 2, best_def_val, height,
                    color='C0', alpha=0.5, zorder=0, edgecolor='gold', linewidth=3)

    if highlight_mode == 'star' and best_y is not None:
        best_tuned_val = ds_df.loc[best_y, "tuned_acc"]
        best_tuned_err = opt_err.iloc[best_y] if pd.notna(opt_err.iloc[best_y]) else 0
        if pd.notna(best_tuned_val):
            x_star = best_tuned_val + best_tuned_err + 0.02
            ax.plot(x_star, best_y + height / 2, marker='*', color='goldenrod',
                    markersize=3, zorder=5, linestyle='None')

    # Formatting Y-Axis
    ax.set_yticks(y)
    if i == 0:
        ax.set_yticklabels(ds_df["display_name"], fontsize=8)
        ax.tick_params(axis='y', pad=Y_TICK_PADDING)
    else:
        ax.tick_params(axis='y', which='both', left=False)

    if i == 0:
        cat_blocks = []
        current_cat = None
        start_idx = 0

        for idx, row in ds_df.iterrows():
            if row['category'] != current_cat:
                if current_cat is not None:
                    line = ax.axhline(idx - 0.5, color='black', lw=0.5, ls='-', alpha=0.2, zorder=1, clip_on=False)
                    line.set_xdata([LINE_EXTENSION, 1.0])
                    for a in axes[1:]:
                        a.axhline(idx - 0.5, color='black', lw=0.5, ls='-', alpha=0.2, zorder=1)
                    cat_blocks.append((current_cat, start_idx, idx - 1))
                current_cat = row['category']
                start_idx = idx
        cat_blocks.append((current_cat, start_idx, len(ds_df) - 1))

        if SHOW_CATEGORY_LABELS:
            for cat, start, end in cat_blocks:
                mid_y = (start + end) / 2.0
                ax.text(CAT_LABEL_X_POS, mid_y, cat, transform=ax.get_yaxis_transform(),
                        fontsize=8, fontweight='bold', va='center', ha='center',
                        color='gray', rotation=90)

    ax.set_title(dataset_labels.get(ds, ds), fontsize=10)
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis="x", linestyle="--", alpha=0.5, zorder=1)

    valid = ds_df["default_acc"].notna() | ds_df["tuned_acc"].notna()
    if valid.any():
        x_min = min((ds_df["default_acc"] - def_err).min(), (ds_df["tuned_acc"] - opt_err).min())
        x_max = max((ds_df["default_acc"] + def_err).max(), (ds_df["tuned_acc"] + opt_err).max())
        margin = (x_max - x_min) * 0.05
        extra_right = 0.03 if highlight_mode == 'star' else 0
        ax.set_xlim(max(0, x_min - margin), min(1.0, x_max + margin + extra_right))

axes[3].legend(
    loc="upper right", fontsize=8, framealpha=0.4,
    borderpad=0.2, handletextpad=0.4, handlelength=1.0, labelspacing=0.3
)
fig.supxlabel("Accuracy", fontsize=9)

# Save
export_dir = os.path.join(current_path, "experiment_files", "export", "fig3", "comparision_unsupervised")
os.makedirs(export_dir, exist_ok=True)
plt.savefig(os.path.join(export_dir, "categorical_comparison.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
import torch


@torch.no_grad()
def get_aggregated_bounds(transform_seq, n_sampling=10000, n_orbit=10000, verbose=True, device="cpu"):
    param_sizes = transform_seq.extract_param_sizes()
    total_dim = sum(param_sizes)

    mins_candidates = []
    maxs_candidates = []

    # --- Method 1: Statistical Sampling (10k samples) ---
    samples = transform_seq.sample_individual(n_sampling).view(-1, total_dim)
    mins_candidates.append(samples.min(dim=0)[0])
    maxs_candidates.append(samples.max(dim=0)[0])

    # --- Method 2: Orbit Sweep ---
    orbit_mins = torch.full((total_dim,), float('inf'), device=device)
    orbit_maxs = torch.full((total_dim,), float('-inf'), device=device)

    curr = 0
    for i, trans in enumerate(transform_seq.transformations):
        domain = transform_seq.domains[i]
        p_size = param_sizes[i]
        for local_d in range(p_size):
            try:
                # orbit returns (N, p_size)
                orb_params = trans.orbit(n_samples=n_orbit, domain=domain, dim=local_d)
                vals = orb_params[:, local_d]
                orbit_mins[curr + local_d] = vals.min()
                orbit_maxs[curr + local_d] = vals.max()
            except Exception:
                pass
        curr += p_size

    mins_candidates.append(orbit_mins)
    maxs_candidates.append(orbit_maxs)

    # --- Method 3: Direct Domain Interpretation & Reprojection ---
    low_list, high_list = [], []
    for i, domain in enumerate(transform_seq.domains):
        p_size = param_sizes[i]
        try:
            # Case 1: Vector domain ((x_min, x_max), (y_min, y_max))
            if isinstance(domain[0], (list, tuple, torch.Tensor)):
                l = torch.tensor([d[0] for d in domain], device=device)
                h = torch.tensor([d[1] for d in domain], device=device)
            # Case 2: Single value (min, max) repeated for all dims
            else:
                l = torch.full((p_size,), domain[0], device=device)
                h = torch.full((p_size,), domain[1], device=device)
            low_list.append(l)
            high_list.append(h)
            print(low_list)
            print(high_list)
        except Exception:
            # Fallback to zero if interpretation fails
            low_list.append(torch.zeros(p_size, device=device))
            high_list.append(torch.zeros(p_size, device=device))

    domain_pts = torch.stack([torch.cat(low_list), torch.cat(high_list)], dim=0)
    # Reproject through correct_param to handle periodic wrapping/clamping
    projected_domain = transform_seq.correct_param(domain_pts, reflect=False)
    mins_candidates.append(projected_domain.min(dim=0)[0])
    maxs_candidates.append(projected_domain.max(dim=0)[0])
    projected_domain = transform_seq.correct_param(domain_pts + 1e-6, reflect=False)
    mins_candidates.append(projected_domain.min(dim=0)[0])
    maxs_candidates.append(projected_domain.max(dim=0)[0])
    projected_domain = transform_seq.correct_param(domain_pts - 1e-6, reflect=False)
    mins_candidates.append(projected_domain.min(dim=0)[0])
    maxs_candidates.append(projected_domain.max(dim=0)[0])

    # --- Method 4: Extreme Value Probe (+/- 1e9) ---
    inf_proxy = 1e9
    extreme_pts = torch.tensor([
        [-inf_proxy] * total_dim,
        [inf_proxy] * total_dim
    ], device=device, dtype=torch.float32)

    projected_extremes = transform_seq.correct_param(extreme_pts, reflect=False)
    mins_candidates.append(projected_extremes.min(dim=0)[0])
    maxs_candidates.append(projected_extremes.max(dim=0)[0])

    # Stack and find the global min/max for each dimension
    all_mins = torch.stack(mins_candidates)
    all_maxs = torch.stack(maxs_candidates)

    # We want the 'outermost' bounds found by any method
    final_mins = all_mins.min(dim=0)[0]
    final_maxs = all_maxs.max(dim=0)[0]

    if verbose:
        print(f"\n--- Robust Aggregated Bounds (Dim 1 check: Pitch) ---")
        for d in range(total_dim):
            print(f"Dim {d}: [{final_mins[d].item():.4f}, {final_maxs[d].item():.4f}]")

    print(mins_candidates)
    print(maxs_candidates)

    return final_mins, final_maxs

In [ ]:
def generate_transformation_meshgrid(transform_seq, grid_size=20, device="cpu"):
    mins, maxs = get_aggregated_bounds(transform_seq, device=device)

    # Generate linspaces based on these final aggregated bounds
    dims = []
    for d in range(len(mins)):
        dims.append(torch.linspace(mins[d], maxs[d], grid_size, device=device))

    grid = torch.meshgrid(*dims, indexing='ij')
    return torch.stack(grid, dim=-1).reshape(-1, len(mins))

In [ ]:
stop

The following visualizes some of the energy landscape. Dataset must be bigger emnist.

In [ ]:
transform_seq_vis = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)
grid_size = 21
mesh_params = generate_transformation_meshgrid(transform_seq_vis, grid_size, device=device)



In [ ]:
# Pick a single datapoint for the landscape evaluation
idx = 0
if dataset == 'bigger_emnist':
    idx = 0

datapoint = next(iter(test_loader))[0][idx:idx + 1].to(device)
true_label = next(iter(test_loader))[1][idx].item()

plt.imshow(datapoint.cpu()[0, 0])

In [ ]:
landscapes_results_dir

In [ ]:
def generate_sliced_transformation_meshgrid(transform_seq, indices=[0, 1], grid_size=20, device="cpu"):
    mins, maxs = get_aggregated_bounds(transform_seq, device=device)
    num_dims = len(mins)

    varied_linspaces = []
    for idx in indices:
        varied_linspaces.append(
            torch.linspace(mins[idx], maxs[idx], grid_size, device=device)
        )

    # Create the mesh for the active indices only
    grid = torch.meshgrid(*varied_linspaces, indexing='ij')
    flat_grid = torch.stack(grid, dim=-1).reshape(-1, len(indices))

    total_points = flat_grid.shape[0]

    # Get identity parameters
    identity_params = transform_seq.get_identity_parameters(batch_size=1)

    # Ensure tensor, correct device, and flattened
    identity_params = identity_params.to(device).reshape(1, -1)

    # Repeat identity for every grid point
    final_params = identity_params.repeat(total_points, 1)

    # Inject the varied parameters into the correct columns
    final_params[:, indices] = flat_grid

    return final_params

In [ ]:
import os
import json
import torch
import numpy as np
import tqdm

# --- New Configuration for Sliced Mesh ---
landscapes_sliced_dir = os.path.join(base_results_dir, "energy_landscapes4")
os.makedirs(landscapes_sliced_dir, exist_ok=True)

transform_seq_vis = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

grid_size_sliced = 81
mesh_params_sliced = generate_sliced_transformation_meshgrid(
    transform_seq_vis,
    indices=[0, 1],
    grid_size=grid_size_sliced,
    device=device
)

total_transforms_sliced = mesh_params_sliced.shape[0]
batch_size_sliced = dataset_info.batch_size_search

# Pick a single datapoint
datapoint = next(iter(test_loader))[0][0:1].to(device)
model.eval().to(device)

for detector in detectors:
    if detector == "laplace_entropy_gridsearch":
        continue

    print(f"\n--- Sliced Energy Calculation: {detector} ---")

    cache_filename = f"energy_sliced_{detector}_grid{grid_size_sliced}.pt"
    cache_path = os.path.join(landscapes_sliced_dir, cache_filename)

    if os.path.exists(cache_path):
        print(f"[*] Found cached sliced energies at {cache_path}.")
        energies_sliced = torch.load(cache_path)
    else:
        # Problem setup
        detector_dir = os.path.join(base_results_dir, detector)
        params_path = os.path.join(detector_dir, "best_params.json")
        best_params = json.load(open(params_path, "r")) if os.path.exists(params_path) else get_default_ood_params(
            detector)

        final_kwargs = {
            "model": model, "train_cache": train_cache, "transform_seq": transform_seq_vis,
            "dataset_info": dataset_info, "architecture": architecture, "device": str(device),
        }
        problem = create_ood_problem(detector, best_params, **final_kwargs)

        energies_list = []

        with torch.no_grad():
            # Progress bar for sliced computation
            for i in tqdm.tqdm(range(0, total_transforms_sliced, batch_size_sliced), desc=f"Sliced {detector}"):
                params_batch = mesh_params_sliced[i: i + batch_size_sliced].to(device)
                curr_bs = params_batch.shape[0]

                x_input = datapoint.clone()
                if x_input.shape[0] != 1 or x_input.dim() < 3:
                    x_input = x_input.unsqueeze(0)

                expansion_shape = [curr_bs] + ([-1] * (x_input.dim() - 1))
                x_rep = x_input.expand(*expansion_shape).contiguous()
                x_trans = problem.transform(x_rep, params_batch)

                try:
                    energy = problem.confidence_module(x_trans)[0]
                except (RuntimeError, TypeError):
                    B, D = x_trans.shape[0], x_trans.shape[-1]
                    x_flat = x_trans.reshape(-1, D)
                    num_elements_per_sample = x_flat.shape[0] // B
                    batch_vec = torch.arange(B, device=device).repeat_interleave(num_elements_per_sample)
                    energy = problem.confidence_module(x_flat, batch=batch_vec)[0]

                # Correctly append inside the loop
                energies_list.append(energy.view(curr_bs).cpu())

            # Save after the loop finishes
            energies_sliced = torch.cat(energies_list, dim=0)
            torch.save(energies_sliced, cache_path)
            print(f"[+] Sliced energies saved. Shape: {energies_sliced.shape}")

print("\nSliced precalculation complete.")

In [ ]:
from src.utils.transformation_problem import TransformationProblem

correct_cache_path = os.path.join(landscapes_sliced_dir, f"correct_vals_grid{grid_size_sliced}.pt")
# Problem shell to use the shared transform logic
problem_shell = TransformationProblem(None, transform_seq_vis)

if os.path.exists(correct_cache_path):
    print(f"[*] Found cached correctness. Loading...")
    correctness_sliced = torch.load(correct_cache_path)
else:
    print(f"[*] Computing correctness for {total_transforms_sliced} transforms...")
    correct_list = []
    with torch.no_grad():
        for i in tqdm.tqdm(range(0, total_transforms_sliced, batch_size_sliced), desc="Correctness"):
            params_batch = mesh_params_sliced[i:i + batch_size_sliced].to(device)
            curr_bs = params_batch.shape[0]
            expansion_shape = [curr_bs] + [-1] * (datapoint.dim() - 1)
            x_rep = datapoint.expand(*expansion_shape).contiguous()
            x_trans = problem_shell.transform(x_rep, params_batch)
            preds = model(x_trans).argmax(dim=1).cpu()
            correct_list.append((preds == true_label).float())
    correctness_sliced = torch.cat(correct_list, dim=0)
    torch.save(correctness_sliced, correct_cache_path)
    print(f"[+] Correctness cached. Shape: {correctness_sliced.shape}")

In [ ]:
# save to figures folder if exists
figure_path3 = os.path.join(current_path, "experiment_files", "export", "fig3", "vis_landscapes", dataset,
                            getattr(dataset_info, "transform_seq_name", "default"))
os.makedirs(figure_path, exist_ok=True)
W = plt_setup_paper()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import os
from scipy.stats import rankdata


def visualize_rank_landscape_overlay(detector_name, grid_size, mesh_params, save_path=None, indices=(0, 1),
                                     flip_correctness=False):
    # Construct paths for energy and correctness data
    e_path = os.path.join(landscapes_sliced_dir, f"energy_sliced_{detector_name}_grid{grid_size}.pt")
    c_path = os.path.join(landscapes_sliced_dir, f"correct_vals_grid{grid_size}.pt")

    if not (os.path.exists(e_path) and os.path.exists(c_path)):
        print(f"Files not found for {detector_name}")
        return

    # 1. Load and Transpose
    energy_raw = torch.load(e_path).view(grid_size, grid_size).numpy().T
    correct_plot = torch.load(c_path).view(grid_size, grid_size).numpy().T

    # rankdata flattens, ranks, then we reshape back.
    # Resulting values are in range [1, N*M]. We normalize to [0, 1].
    energy_rank = rankdata(energy_raw).reshape(energy_raw.shape)
    energy_rank = (energy_rank - 1) / (energy_rank.max() - 1)

    # 2. Handle 0 vs 1 Logic
    if flip_correctness:
        correct_plot = 1 - correct_plot

    # 3. Extract Bounds
    x_coords_raw = mesh_params[:, indices[0]].cpu().numpy()
    y_coords_raw = mesh_params[:, indices[1]].cpu().numpy()
    extent = [x_coords_raw.min(), x_coords_raw.max(), y_coords_raw.min(), y_coords_raw.max()]

    x_coords = np.linspace(x_coords_raw.min(), x_coords_raw.max(), grid_size)
    y_coords = np.linspace(y_coords_raw.min(), y_coords_raw.max(), grid_size)

    fig, ax = plt.subplots(figsize=(W / 2, W * 0.75 / 2))

    # Background Energy Plot (Using Rank) ---
    im = ax.imshow(energy_rank, origin='lower', extent=extent, cmap='plasma', aspect='auto')

    #Overlay Accuracy
    res = 10
    correct_blocky = np.repeat(np.repeat(correct_plot, res, axis=0), res, axis=1)
    contour_color = 'lime' if not flip_correctness else 'red'

    ax.contour(correct_blocky, levels=[0.5], extent=extent,
               colors=contour_color, linewidths=1.0, linestyles='solid')

    # Peak remains at the same location regardless of ranking
    best_idx = np.unravel_index(np.argmax(energy_raw), energy_raw.shape)
    best_y, best_x = y_coords[best_idx[0]], x_coords[best_idx[1]]
    ax.scatter(best_x, best_y, color='cyan', marker='*', s=300,
               edgecolors='black', label='Detector Peak', zorder=10)

    # Aesthetics
    ax.axvline(0, color='white', linestyle='--', alpha=0.3)
    ax.axhline(0, color='white', linestyle='--', alpha=0.3)
    #ax.set_title(f"Detector: {detector_name}")

    #ax.set_xlabel(f"Dim {indices[0]}")
    ax.set_xlabel(f"Rotation")
    #ax.set_ylabel(f"Dim {indices[1]}")
    ax.set_ylabel(f"Shearing")

    ax.legend()

    plt.colorbar(im, label="Confidence Rank (Percentile)")
    plt.tight_layout()

    if save_path:
        os.makedirs(save_path, exist_ok=True)
        filename = f"rank_landscape_{detector_name}_grid{grid_size}.png"
        full_save_path = os.path.join(save_path, filename)
        plt.savefig(full_save_path, dpi=300, bbox_inches='tight')
        print(f"Saved: {full_save_path}")

    plt.show()
    plt.close(fig)


for detector in detectors:
    if detector != "laplace_entropy_gridsearch":
        visualize_rank_landscape_overlay(
            detector,
            grid_size_sliced,
            mesh_params_sliced,
            save_path=figure_path3,
            flip_correctness=False
        )